# Earth Observation Modelling of Informal Settlement Flood Exposure Across East African Cities
## Reproducible pipeline: data preparation → 7 models → explainability → uncertainty → maps → external validation

**Article:** Disproportionate flood exposure of informal settlements in five East African cities (Natural Hazards, submitted)  
**Article authors:** Katabarwa Murenzi Gilbert, Qian Shi (corresponding: qianshi@tongji.edu.cn), Tongji University  
**Cities:** Kigali, Kampala, Nairobi, Dar es Salaam, Bujumbura  
**Runs on:** a local PC (JupyterLab, CPU is enough) **or** Google Colab — detected automatically.  
**Input:** the `Project III` folder (local: `G:\My Drive\Project III` via Google Drive for Desktop; Colab: `MyDrive/Project III`).  
**Output:** one folder, `Project III/M2_Final_Reproducible_Results/`, resumable city by city.

| Cell | Step | Main outputs |
|---|---|---|
| 0 | Setup | environment, paths, checkpoint |
| 1 | Configuration | data paths per city, predictor set, sampling and training settings |
| 2 | Helpers | models (RF, XGBoost, CatBoost, LightGBM, ResNet1D, TabTransformer, Stacking), sampling, spatial CV, ALE |
| 3 | Pre-flight audit | `Tables/Preflight_raster_map.csv` (file used for every predictor), label-raster coding |
| 4 | Training and analysis | 7 models × 5 cities, SHAP, ALE, uncertainty, LOCO, label sanity checks |
| 5 | Summary tables | AUC, spatial-CV AUC, PR-AUC, F1, MCC, Kappa, Brier, thresholds, sampling log |
| 6 | Wall-to-wall maps | probability, 5-class (Jenks) susceptibility and uncertainty GeoTIFFs + QGIS `.qml` styles |
| 7 | Cartographic maps | 300-dpi maps with north arrow, scale bar, graticule, legend |
| 8 | Publication figures | ROC curves, AUC bars, LOCO heatmap, uncertainty bars |
| 9 | External validation | Global Flood Database (Tellman et al., 2021), optional, needs Google Earth Engine |
| 10 | Run manifest | package versions, settings, timings, output inventory |

### Sampling protocol (identical for all cities)
1. Flood and non-flood locations come from each city's flood inventory: binary raster (Kigali, Nairobi, Bujumbura) or inventory CSV (Kampala, Dar es Salaam).
2. Only pixels inside the city boundary **and** covered by every predictor are eligible, so no sample is lost to NoData.
3. Balanced classes, at most 2,500 per class, drawn at random from eligible locations (seed 42).

### Predictors (19)
Aspect, CI, Curvature, DEM, Distance to river, Distance to water, Drainage density, Flow accumulation, NDBI, NDVI, NDWI, Profile curvature, Rainfall, Slope, SPI, STI, Stream density, TRI, TWI.
Removed after audit: `Distance_to_Drainage_Channels` (identical to `Distance_to_River` in all cities) and `FlowDir` (categorical D8 codes). Multicollinearity is then controlled per city with VIF ≤ 10. Rainfall NoData gaps are filled by nearest neighbour on a 1 km grid.

### Evaluation
- **Test set:** spatial-block hold-out, 20 % of samples per class taken as whole ~1 km blocks that never touch training blocks.
- **Tuning:** Bayesian optimisation with spatial-block cross-validation inside the training blocks.
- **Decision threshold:** Youden's J on out-of-fold training predictions (deep models: inner validation split). The test set is never used for tuning, early stopping or thresholds.
- **Explainability:** SHAP (TreeExplainer) and first-order ALE (Apley & Zhu, 2020; implemented here, no external ALE package) on the best tree model; agreement as Spearman ρ.
- **Uncertainty:** inter-model standard deviation of the four tree models, and Monte Carlo dropout (T = 50) for ResNet1D.
- **External validation:** Global Flood Database (MODIS, 250 m, permanent water excluded). Kampala's inventory represents urban pluvial (drainage-driven) flooding, so the fluvial/lacustrine GFD record is flagged as not applicable there.

### How to run
- **Local PC (recommended):** Miniforge environment `m2` (see `requirements_M2_local.txt`), then `jupyter lab` and **Run All**. About 1.5–2.5 h on a 6-core CPU.
- **Colab:** upload this notebook and **Run All**; packages are installed automatically.
- If interrupted, run all cells again: finished cities and maps are skipped.

**Re-run to finish (quota-friendly):** the checkpoint already holds all 5 trained cities, so **Run all** skips training (Cell 4) and only builds the missing maps, figures, external validation, slum maps, Tables 1–2 and supplementary tables. **A CPU runtime is enough — no GPU needed.** Expected time ≈ 30–60 min. Any city that fails is reported and the others continue; re-running a cell skips finished outputs.

**Added for the manuscript tables:** Cell 11 — informal-settlement flood exposure (Table 1 areas, Table 2 chi-square and RFEI); Cell 12 — Supplementary Tables S1 (VIF), S2 (LOCO) and A1 (settings); Cell 10b — slum-probability maps from the Sentinel-2 exports (random forest, 100 trees) with spatial-block accuracy (Table S3) and Figure 2; Cell 11b — Figures 4 and 6a.


In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 0 — Setup
# ═════════════════════════════════════════════════════════════════════
# Works in Google Colab AND on a local PC (Jupyter / VS Code).
# Local PC: install packages once from requirements_M2_local.txt, and point M2_BASE (or the default below)
# to the 'Project III' folder, e.g. via Google Drive for Desktop: G:\\My Drive\\Project III
try:
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm', 'catboost', 'xgboost',
                    'scikit-optimize', 'shap', 'pyproj', 'geopandas', 'jenkspy'], check=False)

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'   # Windows: avoid OpenMP DLL clash between torch and conda-forge libs
import torch                                  # Windows: import torch BEFORE rasterio/GDAL/sklearn to avoid DLL conflicts
import gc, json, warnings, pickle, time, sys, platform
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib, rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import geometry_mask
from rasterio.fill import fillnodata
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split, cross_val_predict, BaseCrossValidator
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, average_precision_score, confusion_matrix, brier_score_loss, cohen_kappa_score, matthews_corrcoef)
from sklearn.base import clone
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import spearmanr
from pyproj import Transformer
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb
import shap, torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import geopandas as gpd
import jenkspy

warnings.filterwarnings('ignore')
np.random.seed(42); torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LOCAL_BASE  = r'G:\My Drive\Project III'     # ← edit if your Drive letter / folder differs
BASE        = Path('/content/drive/MyDrive/Project III') if IN_COLAB else Path(os.environ.get('M2_BASE', LOCAL_BASE))
DATA        = BASE / 'Data'
RESULTS_DIR = BASE / 'M2_Final_Reproducible_Results'
MODELS_DIR  = RESULTS_DIR / 'Models'
TABLES_DIR  = RESULTS_DIR / 'Tables'
FIGURES_DIR = RESULTS_DIR / 'Figures'
MAPS_DIR    = RESULTS_DIR / 'Cartographic_Maps'
TIF_DIR     = RESULTS_DIR / 'TIF_Maps'          # GeoTIFF + .qml sidecars (auto-styled in QGIS)
ROC_DIR     = RESULTS_DIR / 'ROC_data'
EXTVAL_DIR  = RESULTS_DIR / 'External_Validation'
for d in [MODELS_DIR, TABLES_DIR, FIGURES_DIR, MAPS_DIR, TIF_DIR, ROC_DIR, EXTVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FILE = RESULTS_DIR / '_checkpoint.json'

assert DATA.exists(), (f'❌ {DATA} not found — Colab: mount the Drive that holds Project III; '
                       f'local PC: set LOCAL_BASE to your Project III folder')

class NumpyEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.integer):  return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.ndarray):  return o.tolist()
        if isinstance(o, Path):        return str(o)
        return super().default(o)

RESET = False   # True = ignore checkpoint and recompute every city

def _empty_ckpt():
    return {'completed': [], 'all_results': {}, 'xai_summary': [], 'uq_summary': [], 'sampling_log': [],
            'label_checks': [], 'loco_all_files': []}

def load_checkpoint():
    if RESET or not CHECKPOINT_FILE.exists():
        return _empty_ckpt()
    try:
        with open(CHECKPOINT_FILE) as f:
            ck = json.load(f)
        print(f'✅ Resuming — completed: {ck["completed"]}')
        return ck
    except Exception as e:
        print(f'⚠️ Checkpoint unreadable ({e}) — starting fresh')
        return _empty_ckpt()

def save_checkpoint(ck):
    tmp = CHECKPOINT_FILE.with_suffix('.tmp')
    with open(tmp, 'w') as f:
        json.dump(ck, f, cls=NumpyEncoder)
    os.replace(tmp, CHECKPOINT_FILE)          # atomic: never leaves a half-written file
    print(f'  💾 Checkpoint: {ck["completed"]}')

CKPT = load_checkpoint()
T_START = time.time()
print(f'✅ Setup | device={device} | {datetime.now():%Y-%m-%d %H:%M}')

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 1 — City configs (paths verified against Project III tree, 2026-11-02)
# ═════════════════════════════════════════════════════════════════════
CITIES = {
    'Kigali': {
        'csv': DATA/'Kigali'/'Kigali_Flood_Samples_2015_2025.csv',
        'lat_col': 'latitude', 'lon_col': 'longitude', 'label_col': 'label',
        'raster_dir': DATA/'Kigali',
        'label_raster': DATA/'Kigali'/'Kigali_Flood_Label_2015_2025.tif',
        'boundary': DATA/'Kigali'/'Kigali'/'City.shp',
    },
    'Kampala': {
        'csv': DATA/'Kampala'/'Balanced_FloodSamples_withPredictors.csv',
        'lat_col': 'y', 'lon_col': 'x', 'label_col': 'label',
        'has_predictors': True,                     # predictors already extracted in the CSV
        'raster_dir': DATA/'Kampala',               # used for the wall-to-wall map only
        'boundary': DATA/'Kampala'/'kampala.shp'/'city.shp',
    },
    'Nairobi': {
        'csv': DATA/'Nariobi'/'FloodSamples_Nairobi (1).csv',
        'lat_col': 'latitude', 'lon_col': 'longitude', 'label_col': 'label',
        'raster_dir': DATA/'Nariobi',
        'label_raster': DATA/'Nariobi'/'FloodLabels_Nairobi.tif',
        'boundary': DATA/'Nariobi'/'Nariobi.shp'/'nariobi.shp',
    },
    'Dar_es_Salaam': {
        'csv': DATA/'Dar_Salama'/'FloodSamples_withLatLon.csv',
        'lat_col': 'latitude', 'lon_col': 'longitude', 'label_col': 'label',
        'raster_dir': DATA/'Dar_Salama',
        # CSV inventory kept: FloodRaster.tif marks 30 % of the city as flooded and lowered skill (v2 test)
        'boundary': DATA/'Dar_Salama'/'Dar-salama.shp'/'city.shp',
    },
    'Bujumbura': {
        'csv': DATA/'Bujumbura'/'FloodSamples.csv',
        'lat_col': 'latitude', 'lon_col': 'longitude', 'label_col': 'label',
        'raster_dir': DATA/'Bujumbura',
        'label_raster': DATA/'Bujumbura'/'FloodRaster_clean.tif',
        'boundary': DATA/'Bujumbura'/'Bujumbura.Shp'/'city.shp',
    },
}
# If a label raster is not binary, that city automatically falls back to its CSV points.

TARGET_PREDICTORS = [
    'Aspect', 'CI', 'Curvature', 'DEM',
    'Distance_to_River', 'Distance_to_Water', 'Drainage_Density', 'FlowAccum',
    'NDBI', 'NDVI', 'NDWI', 'ProfileCurvature', 'Rainfall', 'Slope',
    'SPI', 'STI', 'Stream_Density', 'TRI', 'TWI',
]
EXCLUDED_PREDICTORS = {
    'Distance_to_Drainage_Channels': 'identical to Distance_to_River in all five cities (diagnostic audit)',
    'FlowDir': 'D8 flow-direction codes (1–128) are categorical, not ordinal',
}
CSV_COLUMN_ALIASES = {'FlowAccu': 'FlowAccum'}          # Kampala CSV naming
RASTER_ALIASES = {
    'DEM': ['Elevation', 'elevation'],
    'FlowAccum': ['FlowAccu', 'FlowAcc', 'Flow_Accumulation'],
    'Rainfall': ['Annual_Rainfall_2025', 'Annual_Rainfall'],
    'NDVI': ['NDVI_2024'], 'NDWI': ['NDWI_2024'], 'NDBI': ['NDBI_2024'],
    'Distance_to_Water': ['DistanceToWater_2024'],
}
FILL_PREDICTORS = ['Rainfall']    # smooth coarse climate layer: NoData filled by nearest neighbour
SEARCH_SUBDIRS  = ['Aligned', 'Resampled', 'resampled', 'New_resampled']   # 'new_resampling' excluded
MIN_TIF_BYTES   = 1000
REF_PREDICTOR   = 'Slope'
MAX_MAP_PIXELS  = 25_000_000

# Sampling
THIN_GAP_M   = 0                 # 0 = no thinning (autocorrelation handled by spatial hold-out)
TEST_FRAC    = 0.2
GEE_PROJECT  = 'flood-mapping-gee-468011'
N_PER_CLASS  = 2500

# Training budget (free tier)
BAYES_N_ITER = 10
DL_EPOCHS    = 20
DL_PATIENCE  = 5
MC_DROPOUT_T = 50
LOCO_MODELS  = ['RF', 'XGBoost', 'CatBoost']
SPATIAL_BLOCK_DEG = 0.01          # ≈1.1 km spatial CV blocks

CLASS_METHOD = 'jenks'            # 'jenks' (natural breaks) or 'equal' (0.2 steps)

print(f'✅ Configs loaded | {len(TARGET_PREDICTORS)} predictors | excluded: {list(EXCLUDED_PREDICTORS)}')

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 2 — Models + helpers
# ═════════════════════════════════════════════════════════════════════
class SpatialBlockCV(BaseCrossValidator):
    # True spatial-block CV: samples are grouped into square grid cells on their coordinates,
    # and whole cells are assigned to folds (no cell is split between train and validation).
    def __init__(self, coords, n_splits=3, block=SPATIAL_BLOCK_DEG, seed=42):
        self.coords = np.asarray(coords, dtype=float)
        self.n_splits, self.seed = n_splits, seed
        self.block = 1000.0 if np.nanmax(np.abs(self.coords)) > 360 else block
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits
    def split(self, X, y=None, groups=None):
        cells = np.floor(self.coords / self.block).astype(np.int64)
        _, bid = np.unique(cells, axis=0, return_inverse=True)
        bid = bid.ravel()
        ub = np.unique(bid)
        np.random.RandomState(self.seed).shuffle(ub)
        for fold in np.array_split(ub, self.n_splits):
            te = np.isin(bid, fold)
            yield np.where(~te)[0], np.where(te)[0]


class ResNet1DBlock(nn.Module):
    def __init__(self, d, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.BatchNorm1d(d), nn.Linear(d, d), nn.ReLU(), nn.Dropout(dropout),
                                 nn.BatchNorm1d(d), nn.Linear(d, d), nn.ReLU(), nn.Dropout(dropout))
    def forward(self, x): return x + self.net(x)

class ResNet1D(nn.Module):
    def __init__(self, n_feat, d=128, blocks=3, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_feat, d), *[ResNet1DBlock(d, drop) for _ in range(blocks)],
                                 nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

class TabTransformer(nn.Module):
    def __init__(self, n_feat, d=64, heads=4, layers=3, drop=0.1):
        super().__init__()
        self.embed = nn.Linear(1, d)
        self.pos = nn.Parameter(torch.randn(1, n_feat, d) * 0.02)
        enc = nn.TransformerEncoderLayer(d, heads, d * 4, drop, batch_first=True)
        self.tf = nn.TransformerEncoder(enc, layers)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Flatten(), nn.Linear(n_feat * d, 128),
                                  nn.ReLU(), nn.Dropout(drop), nn.Linear(128, 1))
    def forward(self, x):
        return self.head(self.tf(self.embed(x.unsqueeze(-1)) + self.pos)).squeeze(-1)


# ---------- raster I/O ----------
def find_raster(raster_dir, name):
    if raster_dir is None:
        return None
    raster_dir = Path(raster_dir)
    aliases = [name] + [a for a in RASTER_ALIASES.get(name, []) if a != name]
    dirs = [raster_dir / s for s in SEARCH_SUBDIRS if (raster_dir / s).is_dir()] + [raster_dir]
    for d in dirs:
        for a in aliases:
            for pat in (f'{a}.tif', f'Resampled_{a}.tif', f'{a}_resampled.tif', f'resampled_{a}.tif'):
                p = d / pat
                if p.exists() and p.stat().st_size >= MIN_TIF_BYTES:
                    return p
    return None

def res_m(transform, crs, lat=0.0):
    r = abs(transform.a)
    return r * 111320 * np.cos(np.radians(lat)) if (crs is not None and crs.is_geographic) else r

def fill_coarse(arr, rm):
    # Nearest-neighbour gap filling for smooth climate layers, done on a ~1 km copy for speed.
    if not np.isnan(arr).any():
        return arr
    k = max(1, int(round(1000 / max(rm, 1e-6))))
    c = arr[::k, ::k].astype(np.float32).copy()
    if np.isnan(c).all():
        return arr
    c = fillnodata(c, mask=(~np.isnan(c)).astype('uint8'), max_search_distance=float(max(c.shape)))
    ri = np.minimum(np.arange(arr.shape[0]) // k, c.shape[0] - 1)
    ci = np.minimum(np.arange(arr.shape[1]) // k, c.shape[1] - 1)
    out = arr.copy()
    rr, cc = np.nonzero(np.isnan(out))
    out[rr, cc] = c[ri[rr], ci[cc]]
    return out

import shutil, tempfile
LOCAL_CACHE = Path(tempfile.gettempdir()) / 'm2_raster_cache'

def _read1(path):
    with rasterio.open(path) as s:
        a = s.read(1).astype(np.float32)
        if s.nodata is not None:
            a[a == s.nodata] = np.nan
        a[~np.isfinite(a)] = np.nan
        return a, s.transform, s.crs, (s.bounds.top + s.bounds.bottom) / 2

def safe_read(path):
    # Cloud-synced drives (Colab Drive / Drive for Desktop) can return a partially downloaded file.
    # On a read error the file is copied to local disk and read again (up to 3 attempts).
    last = None
    for attempt in range(3):
        try:
            src = path if attempt == 0 else LOCAL_CACHE / f'{attempt}_{Path(path).name}'
            if attempt > 0:
                LOCAL_CACHE.mkdir(parents=True, exist_ok=True); time.sleep(3 * attempt)
                shutil.copyfile(path, src)
            out = _read1(src)
            if attempt > 0:
                os.remove(src)
            return out
        except Exception as e:
            last = e
            print(f'      ⚠️ read failed ({Path(path).name}, attempt {attempt + 1}/3): {str(e)[:90]}')
    raise last

def read_band(path, fill=False):
    a, tr, crs, lat = safe_read(path)
    if fill:
        a = fill_coarse(a, res_m(tr, crs, lat))
    return a, tr, crs

def _to_crs(lons, lats, crs):
    if crs is not None and str(crs).upper() != 'EPSG:4326':
        return Transformer.from_crs('EPSG:4326', crs, always_xy=True).transform(lons, lats)
    return lons, lats

def extract_from_rasters(lats, lons, raster_dir):
    names, vals = [], []
    for pred in TARGET_PREDICTORS:
        p = find_raster(raster_dir, pred)
        if p is None:
            continue
        a, tr, crs = read_band(p, pred in FILL_PREDICTORS)
        xs, ys = _to_crs(np.asarray(lons), np.asarray(lats), crs)
        r, c = rasterio.transform.rowcol(tr, xs, ys)
        r, c = np.asarray(r), np.asarray(c)
        ok = (r >= 0) & (r < a.shape[0]) & (c >= 0) & (c < a.shape[1])
        v = np.full(len(r), np.nan, np.float32); v[ok] = a[r[ok], c[ok]]
        del a
        if np.all(np.isnan(v)):
            print(f'      ⚠️ {pred}: all-NoData at sample points ({p.name}) — skipped'); continue
        names.append(pred); vals.append(v)
    if not names:
        return None, None
    return np.column_stack(vals), names


# ---------- sampling ----------
def _spatial_thin(rows, cols, gap, rng):
    # one pixel per gap×gap cell, random choice within the cell (vectorised)
    perm = rng.permutation(len(rows))
    keys = (rows[perm] // gap).astype(np.int64) * 10_000_000 + (cols[perm] // gap).astype(np.int64)
    _, first = np.unique(keys, return_index=True)
    return perm[first]

def decode_label(lab, nd):
    lab = lab.astype(np.float64)
    if nd is not None:
        lab[lab == nd] = np.nan
    u = np.unique(lab[np.isfinite(lab)])
    if len(u) == 2 and set(u.tolist()) == {0.0, 1.0}:
        return lab == 1, lab == 0, 'binary 0/1'
    if len(u) == 2:
        return lab == u.max(), lab == u.min(), f'binary {u.min():g}/{u.max():g} (higher = flood)'
    if len(u) == 1 and u[0] != 0:
        return lab == u[0], ~np.isfinite(lab), f'single value {u[0]:g} = flood; NoData = non-flood'
    raise ValueError(f'not a binary label raster ({len(u)} distinct values, e.g. {u[:6].tolist()})')

def coverage_mask(shape, ltr, lcrs, raster_dir, boundary):
    # eligible pixels = inside city boundary AND valid in every predictor
    h, w = shape
    inside = ~geometry_mask(gpd.read_file(boundary).to_crs(lcrs).geometry, (h, w), ltr)
    m = inside.copy()
    for pred in TARGET_PREDICTORS:
        p = find_raster(raster_dir, pred)
        if p is None:
            continue
        a, tr, crs = read_band(p, pred in FILL_PREDICTORS)
        dst = np.full((h, w), np.nan, np.float32)
        reproject(a, dst, src_transform=tr, src_crs=crs, src_nodata=np.nan,
                  dst_transform=ltr, dst_crs=lcrs, dst_nodata=np.nan, resampling=Resampling.nearest)
        valid = ~np.isnan(dst); del a, dst
        cov = (valid & inside).sum() / max(inside.sum(), 1)
        if cov < 0.5:
            print(f'      ⚠️ {pred} covers only {cov:.0%} of the city — not used to restrict sampling')
            continue
        m &= valid
    return m, inside

def sample_from_label_raster(cfg, seed=42):
    lr = cfg['label_raster']
    with rasterio.open(lr) as s:
        lab, ltr, lcrs, nd = s.read(1), s.transform, s.crs, s.nodata
        lat_c = (s.bounds.top + s.bounds.bottom) / 2
    flood, dry, coding = decode_label(lab, nd); del lab
    cov, inside = coverage_mask(flood.shape, ltr, lcrs, cfg['raster_dir'], cfg['boundary'])
    log = {'Label_source': Path(lr).name, 'Label_coding': coding,
           'Flood_px_total': int(flood.sum()), 'Flood_px_in_city': int((flood & inside).sum())}
    flood &= cov; dry &= cov
    log.update({'Flood_px_eligible': int(flood.sum()), 'Dry_px_eligible': int(dry.sum())})
    gap = max(1, int(round(THIN_GAP_M / res_m(ltr, lcrs, lat_c))))
    rng = np.random.RandomState(seed)
    f = np.argwhere(flood); d = np.argwhere(dry); del flood, dry
    f = f[_spatial_thin(f[:, 0], f[:, 1], gap, rng)]
    d = d[_spatial_thin(d[:, 0], d[:, 1], gap, rng)]
    n = min(N_PER_CLASS, len(f), len(d))
    log.update({'Thin_gap_px': gap, 'Flood_after_thin': len(f), 'Dry_after_thin': len(d), 'N_per_class': n})
    sel = np.vstack([f[rng.choice(len(f), n, replace=False)], d[rng.choice(len(d), n, replace=False)]])
    y = np.r_[np.ones(n), np.zeros(n)].astype(np.int32)
    xs, ys = rasterio.transform.xy(ltr, sel[:, 0], sel[:, 1])
    xs, ys = np.asarray(xs), np.asarray(ys)
    if lcrs is not None and str(lcrs).upper() != 'EPSG:4326':
        xs, ys = Transformer.from_crs(lcrs, 'EPSG:4326', always_xy=True).transform(xs, ys)
    print(f'    {coding} | flood px: {log["Flood_px_total"]:,} total → {log["Flood_px_eligible"]:,} eligible '
          f'| balanced {n}+{n}' + (f' (after {THIN_GAP_M} m thinning)' if THIN_GAP_M > 0 else ''))
    return np.asarray(ys), np.asarray(xs), y, log

def thin_points(lon, lat, y, gap_m=THIN_GAP_M, seed=42):
    if gap_m <= 0:
        return np.arange(len(y))
    cell = gap_m / 111320 if np.nanmax(np.abs(np.r_[lon, lat])) <= 360 else gap_m
    perm = np.random.RandomState(seed).permutation(len(y))
    keys = (np.floor(lon[perm] / cell).astype(np.int64) * 10_000_000 + np.floor(lat[perm] / cell).astype(np.int64)) * 2 + y[perm]
    _, first = np.unique(keys, return_index=True)
    return np.sort(perm[first])

def spatial_holdout(coords, y, test_frac=TEST_FRAC, block=SPATIAL_BLOCK_DEG, seed=42):
    # Whole ~1 km blocks go to the test set until each class reaches test_frac of its samples.
    coords = np.asarray(coords, float)
    b = 1000.0 if np.nanmax(np.abs(coords)) > 360 else block
    _, bid = np.unique(np.floor(coords / b).astype(np.int64), axis=0, return_inverse=True)
    bid = bid.ravel()
    ub = np.unique(bid); np.random.RandomState(seed).shuffle(ub)
    te = np.zeros(len(y), bool)
    need = {c: test_frac * (y == c).sum() for c in (0, 1)}
    for blk in ub:
        have = {c: ((y == c) & te).sum() for c in (0, 1)}
        if all(have[c] >= need[c] for c in (0, 1)):
            break
        m = bid == blk
        if any(have[c] < need[c] and ((y == c) & m).any() for c in (0, 1)):
            te |= m
    return np.where(~te)[0], np.where(te)[0]

def balance_idx(idx, y, cap=N_PER_CLASS, seed=42):
    rng = np.random.RandomState(seed)
    p, n = idx[y[idx] == 1], idx[y[idx] == 0]
    k = min(len(p), len(n), cap)
    return np.sort(np.r_[rng.choice(p, k, replace=False), rng.choice(n, k, replace=False)])

def load_city_data(city, cfg):
    # Returns X, y, feature_names, coords (N×2 lon/lat), sampling log
    df = pd.read_csv(cfg['csv'])
    lc = cfg['label_col']
    log = {'City': city}

    if cfg.get('has_predictors'):
        df = df.rename(columns=CSV_COLUMN_ALIASES)
        excl = {lc, cfg['lat_col'], cfg['lon_col'], 'system:index', '.geo'} | set(EXCLUDED_PREDICTORS)
        fcols = [c for c in df.columns if c not in excl and pd.api.types.is_numeric_dtype(df[c])]
        y = df[lc].values.astype(np.int32)
        lon, lat = df[cfg['lon_col']].values.astype(float), df[cfg['lat_col']].values.astype(float)
        keep = balance_idx(thin_points(lon, lat, y), y)
        X = df[fcols].values[keep].astype(np.float32); y = y[keep]
        coords = np.column_stack([lon[keep], lat[keep]])
        ok = ~np.isnan(X).any(1); X, y, coords = X[ok], y[ok], coords[ok]
        log.update({'Label_source': cfg['csv'].name, 'Label_coding': 'CSV label', 'N_raw': len(df),
                    'N_per_class': int(y.sum())})
        print(f'    CSV {cfg["csv"].name}: {len(df):,} rows → {len(y):,} balanced samples')
        log['N_final'] = len(y)
        return X, y, fcols, coords, log

    lats = lons = y = None
    if cfg.get('label_raster') is not None and Path(cfg['label_raster']).exists():
        try:
            lats, lons, y, slog = sample_from_label_raster(cfg); log.update(slog)
        except Exception as e:
            print(f'    ⚠️ label raster unusable ({e}) — falling back to CSV points')
    if y is None:
        y0 = df[lc].values.astype(np.int32)
        lon0, lat0 = df[cfg['lon_col']].values.astype(float), df[cfg['lat_col']].values.astype(float)
        keep = balance_idx(thin_points(lon0, lat0, y0), y0)
        lats, lons, y = lat0[keep], lon0[keep], y0[keep]
        log.update({'Label_source': cfg['csv'].name, 'Label_coding': 'CSV label', 'N_raw': len(df),
                    'N_per_class': int(y.sum())})
        print(f'    CSV {cfg["csv"].name}: {len(df):,} rows → {len(y):,} balanced samples')

    X, names = extract_from_rasters(lats, lons, cfg['raster_dir'])
    if X is None:
        print(f'    ❌ no rasters in {cfg["raster_dir"]}')
        return None, None, None, None, log
    ok = ~np.isnan(X).any(1)
    if (~ok).sum():
        print(f'    Dropped {(~ok).sum()} NoData points ({(~ok).mean()*100:.1f}%)')
    X, y = X[ok], y[ok]
    coords = np.column_stack([lons, lats])[ok]
    log['N_final'] = int(len(y))
    print(f'    → {X.shape[0]} samples ({int(y.sum())} flood) × {X.shape[1]} predictors')
    return X.astype(np.float32), y, names, coords, log


# ---------- modelling ----------
def vif_filter(X, names, thresh=10.0):
    df = pd.DataFrame(X, columns=names)
    df = df.loc[:, df.std() > 1e-8]
    removed = []
    while df.shape[1] > 2:
        v = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
        if max(v) <= thresh:
            break
        w = df.columns[int(np.argmax(v))]
        removed.append(w); df = df.drop(columns=[w])
    return list(df.columns), removed

def ale_1d(predict, X, j, n_bins=10):
    # First-order accumulated local effects (Apley & Zhu, 2020) on quantile bins, centred.
    x = X[:, j]
    q = np.unique(np.quantile(x, np.linspace(0, 1, n_bins + 1)))
    if len(q) < 3:
        return None
    idx = np.clip(np.searchsorted(q, x, side='right') - 1, 0, len(q) - 2)
    eff, cnt = np.zeros(len(q) - 1), np.zeros(len(q) - 1)
    for k in range(len(q) - 1):
        m = idx == k
        if not m.any():
            continue
        lo, hi = X[m].copy(), X[m].copy()
        lo[:, j], hi[:, j] = q[k], q[k + 1]
        eff[k] = (predict(hi) - predict(lo)).mean(); cnt[k] = m.sum()
    ale = np.r_[0.0, np.cumsum(eff)]
    mid = (ale[:-1] + ale[1:]) / 2
    return ale - (mid * cnt).sum() / cnt.sum()

HYDRO_EXPECT = {'DEM': '-', 'Slope': '-', 'TWI': '+', 'NDWI': '+', 'Distance_to_Water': '-',
                'Distance_to_River': '-', 'FlowAccum': '+'}

def label_sanity(city, X, y, names):
    # Median of each hydrological predictor for flood vs non-flood samples, against physical expectation.
    rows = []
    for f, e in HYDRO_EXPECT.items():
        if f not in names:
            continue
        v = X[:, names.index(f)]
        d = float(np.median(v[y == 1]) - np.median(v[y == 0]))
        rows.append({'City': city, 'Predictor': f, 'Median_nonflood': float(np.median(v[y == 0])),
                     'Median_flood': float(np.median(v[y == 1])), 'Flood_minus_nonflood': d, 'Expected_sign': e,
                     'Consistent': bool((e == '+' and d > 0) or (e == '-' and d < 0))})
    return rows

def youden(y, p):
    fpr, tpr, t = roc_curve(y, p)
    return float(np.clip(t[int(np.argmax(tpr - fpr))], 0, 1))

def eval_model(yt, yprob, name, thr=0.5):
    yp = (yprob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    return {'Model': name, 'Threshold': float(thr),
            'AUC': float(roc_auc_score(yt, yprob)),
            'PR_AUC': float(average_precision_score(yt, yprob)),
            'Acc': float(accuracy_score(yt, yp)),
            'Prec': float(precision_score(yt, yp, zero_division=0)),
            'Rec': float(recall_score(yt, yp, zero_division=0)),
            'Spec': float(tn / (tn + fp)) if (tn + fp) else 0.0,
            'F1': float(f1_score(yt, yp, zero_division=0)),
            'Kappa': float(cohen_kappa_score(yt, yp)),
            'MCC': float(matthews_corrcoef(yt, yp)),
            'Brier': float(brier_score_loss(yt, yprob)),
            'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)}

def train_deep(model, Xtr, ytr, Xva, yva, epochs=DL_EPOCHS, lr=1e-3, bs=4096, pos_weight=1.0):
    # Early stopping on an INNER validation split (never the test set)
    model = model.to(device)
    tdl = DataLoader(TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.float32)), bs, shuffle=True)
    Xv = torch.tensor(Xva, dtype=torch.float32).to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], dtype=torch.float32, device=device))
    best, best_st, wait = -1, None, 0
    for _ in range(epochs):
        model.train()
        for xb, yb in tdl:
            opt.zero_grad(); crit(model(xb.to(device)), yb.to(device)).backward(); opt.step()
        sch.step(); model.eval()
        with torch.no_grad():
            pv = torch.sigmoid(model(Xv)).cpu().numpy()
        a = roc_auc_score(yva, pv)
        if a > best:
            best, wait = a, 0
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= DL_PATIENCE:
                break
    model.load_state_dict(best_st); model.to(device).eval()
    with torch.no_grad():
        pv = torch.sigmoid(model(Xv)).cpu().numpy()
    return model, pv

print('✅ Helpers loaded')

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 3 — Pre-flight audit: which file feeds every predictor + label-raster coding
# ═════════════════════════════════════════════════════════════════════
rows, problems = [], []
for city, cfg in CITIES.items():
    for key in ['csv', 'label_raster', 'boundary']:
        p = cfg.get(key)
        if p is not None and not Path(p).exists():
            problems.append(f'{city}: missing {key} → {p}')
    for pred in TARGET_PREDICTORS:
        p = find_raster(cfg.get('raster_dir'), pred)
        meta = {}
        if p is not None:
            with rasterio.open(p) as s:
                meta = {'CRS': str(s.crs), 'Width': s.width, 'Height': s.height,
                        'Res_m': round(res_m(s.transform, s.crs, (s.bounds.top + s.bounds.bottom) / 2), 1)}
        rows.append({'City': city, 'Predictor': pred,
                     'File': str(p.relative_to(DATA)) if p else 'NOT FOUND',
                     'Size_MB': round(p.stat().st_size / 1e6, 2) if p else None, **meta})
pf = pd.DataFrame(rows)
pf.to_csv(TABLES_DIR / 'Preflight_raster_map.csv', index=False)
print(pf.pivot(index='Predictor', columns='City', values='File').to_string())

print('\nLABEL RASTERS')
for city, cfg in CITIES.items():
    lr = cfg.get('label_raster')
    if lr is None or not Path(lr).exists():
        print(f'  {city}: CSV labels ({cfg["csv"].name})'); continue
    with rasterio.open(lr) as s:
        a = s.read(1); nd = s.nodata
        r = res_m(s.transform, s.crs, (s.bounds.top + s.bounds.bottom) / 2)
    vals, cnt = np.unique(a, return_counts=True)
    top = ', '.join(f'{v:g}:{c:,}' for v, c in sorted(zip(vals, cnt), key=lambda t: -t[1])[:6])
    try:
        _, _, coding = decode_label(a, nd); verdict = f'✅ {coding}'
    except ValueError as e:
        verdict = f'⚠️ {e} → CSV fallback'
    print(f'  {city}: {Path(lr).name} | {a.shape[1]}×{a.shape[0]} @ {r:.0f} m | nodata={nd} | values {top} | {verdict}')
    del a

if problems:
    print('\n❌ PROBLEMS:'); [print('  ', p) for p in problems]
    raise SystemExit('Fix the missing paths above before continuing.')
print('\n✅ All CSVs, label rasters and boundaries found')

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 4 — MASTER LOOP: 7 models × 5 cities + SHAP + ALE + UQ + LOCO (checkpointed per city)
# ═════════════════════════════════════════════════════════════════════
ALL_RESULTS  = CKPT.get('all_results', {})
XAI_SUMMARY  = CKPT.get('xai_summary', [])
UQ_SUMMARY   = CKPT.get('uq_summary', [])
SAMPLING_LOG = CKPT.get('sampling_log', [])
LABEL_CHECKS = CKPT.get('label_checks', [])
_loco_files = [TABLES_DIR / Path(str(f).replace('\\', '/')).name for f in CKPT.get('loco_all_files', [])]
LOCO_ALL = [pd.read_csv(f) for f in _loco_files if f.exists()]

TREE_SPACES = {
    'RF': (RandomForestClassifier(random_state=42, n_jobs=-1),
           {'n_estimators': Integer(200, 500), 'max_depth': Integer(5, 25),
            'min_samples_split': Integer(2, 15), 'max_features': Categorical(['sqrt', 'log2', 0.5])}),
    'XGBoost': (xgb.XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss'),
                {'n_estimators': Integer(200, 500), 'learning_rate': Real(0.01, 0.2, prior='log-uniform'),
                 'max_depth': Integer(3, 8), 'subsample': Real(0.7, 1.0), 'colsample_bytree': Real(0.7, 1.0)}),
    'CatBoost': (CatBoostClassifier(random_seed=42, verbose=0, allow_writing_files=False),
                 {'iterations': Integer(200, 500), 'learning_rate': Real(0.01, 0.2, prior='log-uniform'),
                  'depth': Integer(3, 8), 'l2_leaf_reg': Real(0.1, 10, prior='log-uniform')}),
    'LightGBM': (lgb.LGBMClassifier(random_state=42, n_jobs=1, verbose=-1),
                 {'n_estimators': Integer(200, 500), 'learning_rate': Real(0.01, 0.2, prior='log-uniform'),
                  'max_depth': Integer(3, 8), 'num_leaves': Integer(20, 100),
                  'subsample': Real(0.7, 1.0), 'colsample_bytree': Real(0.7, 1.0)}),
}

for city, cfg in CITIES.items():
    if city in CKPT['completed']:
        print(f'\n⏩ {city} already done'); continue
    t0 = time.time()
    print(f'\n{"="*70}\n🏙️  {city}\n{"="*70}')

    X_raw, y, feat_names, coords, slog = load_city_data(city, cfg)
    if X_raw is None:
        continue
    SAMPLING_LOG = [r for r in SAMPLING_LOG if r.get('City') != city] + [slog]
    lc = label_sanity(city, X_raw, y, feat_names)
    LABEL_CHECKS = [r for r in LABEL_CHECKS if r['City'] != city] + lc
    print(f'  Label sanity: {sum(r["Consistent"] for r in lc)}/{len(lc)} hydrological checks consistent')

    kept, removed = vif_filter(X_raw, feat_names)
    X = X_raw[:, [feat_names.index(f) for f in kept]]
    print(f'  VIF≤10: kept {len(kept)} | removed {removed}')

    tr_i, te_i = spatial_holdout(coords, y)
    Xtr, Xte, ytr, yte, ctr, cte = X[tr_i], X[te_i], y[tr_i], y[te_i], coords[tr_i], coords[te_i]
    print(f'  Spatial hold-out: {len(te_i)} test / {len(tr_i)} train samples, test {yte.mean():.0%} flood, disjoint ~1 km blocks')
    scaler = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
    joblib.dump({'scaler': scaler, 'features': kept}, MODELS_DIR / f'{city}_scaler_features.pkl')
    cv = SpatialBlockCV(ctr, n_splits=3)

    metrics, tree_models, oof, roc_store = [], {}, {}, {}

    # ---- 4 tree models: spatial-CV Bayesian tuning, Youden threshold from out-of-fold predictions ----
    for mname, (est, space) in TREE_SPACES.items():
        print(f'  ⚙️  {mname}...', end=' ', flush=True)
        try:
            opt = BayesSearchCV(est, space, n_iter=BAYES_N_ITER, cv=cv, scoring='roc_auc',
                                random_state=42, n_jobs=-1, verbose=0).fit(Xtr, ytr)
            m = opt.best_estimator_; tree_models[mname] = m
            oof[mname] = cross_val_predict(clone(m), Xtr, ytr, cv=cv, method='predict_proba')[:, 1]
            thr = youden(ytr, oof[mname])
            p = m.predict_proba(Xte)[:, 1]; roc_store[mname] = p
            met = eval_model(yte, p, mname, thr)
            met['CV_AUC'] = float(opt.best_score_)
            met['Best_params'] = json.dumps(dict(opt.best_params_), cls=NumpyEncoder)
            metrics.append(met)
            joblib.dump(m, MODELS_DIR / f'{city}_{mname}.pkl')
            print(f'AUC={met["AUC"]:.4f} | spatial-CV={opt.best_score_:.4f} | F1={met["F1"]:.3f} @t={thr:.2f}')
        except Exception as e:
            print(f'❌ {e}')

    # ---- deep models: inner validation split, class weighting, Youden threshold on validation ----
    Xf, Xv, yf, yv = train_test_split(Xtr_s, ytr, test_size=0.15, random_state=42, stratify=ytr)
    pw = float((yf == 0).sum() / max((yf == 1).sum(), 1))
    for dname, cls in [('ResNet1D', ResNet1D), ('TabTransformer', TabTransformer)]:
        print(f'  ⚙️  {dname}...', end=' ', flush=True)
        try:
            net, pv = train_deep(cls(Xtr_s.shape[1]), Xf, yf, Xv, yv, pos_weight=pw)
            thr = youden(yv, pv)
            with torch.no_grad():
                p = torch.sigmoid(net(torch.tensor(Xte_s, dtype=torch.float32).to(device))).cpu().numpy()
            roc_store[dname] = p
            met = eval_model(yte, p, dname, thr); metrics.append(met)
            torch.save(net.state_dict(), MODELS_DIR / f'{city}_{dname}.pth')
            print(f'AUC={met["AUC"]:.4f} | F1={met["F1"]:.3f} @t={thr:.2f}')
        except Exception as e:
            print(f'❌ {e}')

    # ---- stacking: meta-learner on out-of-fold base predictions ----
    if len(oof) >= 3:
        print('  ⚙️  Stacking...', end=' ', flush=True)
        try:
            names_b = list(oof)
            meta_tr = np.column_stack([oof[n] for n in names_b])
            meta_te = np.column_stack([tree_models[n].predict_proba(Xte)[:, 1] for n in names_b])
            lr = LogisticRegression(random_state=42, max_iter=500).fit(meta_tr, ytr)
            thr = youden(ytr, cross_val_predict(LogisticRegression(random_state=42, max_iter=500),
                                                meta_tr, ytr, cv=cv, method='predict_proba')[:, 1])
            p = lr.predict_proba(meta_te)[:, 1]; roc_store['Stacking'] = p
            met = eval_model(yte, p, 'Stacking', thr); metrics.append(met)
            joblib.dump({'model': lr, 'base_order': names_b}, MODELS_DIR / f'{city}_Stacking_LR.pkl')
            print(f'AUC={met["AUC"]:.4f} | F1={met["F1"]:.3f} @t={thr:.2f}')
        except Exception as e:
            print(f'❌ {e}')

    np.savez_compressed(ROC_DIR / f'{city}.npz', y_true=yte, **roc_store)

    # ---- XAI: SHAP + ALE on the best tree model ----
    if tree_models:
        best_tree = max(tree_models, key=lambda n: next(m['AUC'] for m in metrics if m['Model'] == n))
        bm = tree_models[best_tree]; Xsub = Xte[:500]; shap_rank = None
        print(f'  🔬 SHAP ({best_tree})...', end=' ', flush=True)
        try:
            sv = shap.TreeExplainer(bm, data=shap.sample(Xtr, 100)).shap_values(Xsub, check_additivity=False)
            if isinstance(sv, list): sv = sv[1]
            if sv.ndim == 3: sv = sv[:, :, 1]
            plt.figure(figsize=(10, 7)); shap.summary_plot(sv, Xsub, feature_names=kept, show=False)
            plt.title(f'{city.replace("_", " ")} — SHAP ({best_tree})'); plt.tight_layout()
            plt.savefig(FIGURES_DIR / f'{city}_SHAP.png', dpi=300, bbox_inches='tight'); plt.close('all')
            shap_rank = pd.DataFrame({'Feature': kept, 'Mean_abs_SHAP': np.abs(sv).mean(0)}).sort_values('Mean_abs_SHAP', ascending=False)
            shap_rank.to_csv(TABLES_DIR / f'{city}_SHAP_ranking.csv', index=False)
            print(f'top3={list(shap_rank.Feature.head(3))}')
        except Exception as e:
            print(f'❌ {e}')
        print(f'  🔬 ALE ({best_tree})...', end=' ', flush=True)
        try:
            Xa = Xtr[:1000]
            ale_idx = [i for i in range(len(kept)) if len(np.unique(Xa[:, i])) >= 5]   # ALE needs ≥2 bins
            pred = lambda x: bm.predict_proba(x)[:, 1]
            ale_vals = {}
            for j in ale_idx:
                a = ale_1d(pred, Xa, j)
                if a is not None:
                    ale_vals[kept[j]] = float(np.mean(np.abs(a)))
            ale_rank = pd.DataFrame({'Feature': kept, 'Mean_abs_ALE': [ale_vals.get(f, 0.0) for f in kept]}
                                    ).sort_values('Mean_abs_ALE', ascending=False)
            ale_rank.to_csv(TABLES_DIR / f'{city}_ALE_ranking.csv', index=False)
            if shap_rank is not None:
                mg = shap_rank.merge(ale_rank, on='Feature')
                rho, pval = spearmanr(mg['Mean_abs_SHAP'], mg['Mean_abs_ALE'])
                XAI_SUMMARY = [r for r in XAI_SUMMARY if r['City'] != city] + \
                              [{'City': city, 'Model': best_tree, 'Spearman_rho': float(rho), 'p_value': float(pval)}]
                print(f'SHAP↔ALE ρ={rho:.3f}')
            else:
                print('done')
        except Exception as e:
            print(f'❌ {e}')

    # ---- UQ: inter-model σ + MC Dropout ----
    if len(tree_models) >= 2:
        P = np.stack([m.predict_proba(Xte)[:, 1] for m in tree_models.values()])
        s = P.std(0)
        uq = {'City': city, 'N_models': len(tree_models), 'Mean_sigma': float(s.mean()), 'Max_sigma': float(s.max()),
              'Pct_sigma_gt_0.1': float(np.mean(s > 0.1) * 100), 'Ensemble_Brier': float(brier_score_loss(yte, P.mean(0)))}
        rn_path = MODELS_DIR / f'{city}_ResNet1D.pth'
        if rn_path.exists():
            rn = ResNet1D(Xte_s.shape[1]).to(device)
            rn.load_state_dict(torch.load(rn_path, map_location=device)); rn.train()
            Xt = torch.tensor(Xte_s, dtype=torch.float32).to(device)
            with torch.no_grad():
                mc = np.stack([torch.sigmoid(rn(Xt)).cpu().numpy() for _ in range(MC_DROPOUT_T)])
            mm, ms = mc.mean(0), mc.std(0)
            ent = -mm * np.log(mm + 1e-8) - (1 - mm) * np.log(1 - mm + 1e-8)
            uq.update({'MC_Mean_sigma': float(ms.mean()), 'MC_Max_sigma': float(ms.max()),
                       'MC_Median_sigma': float(np.median(ms)), 'MC_Mean_entropy': float(ent.mean()),
                       'MC_Pct_sigma_gt_0.1': float(np.mean(ms > 0.1) * 100)})
        UQ_SUMMARY = [r for r in UQ_SUMMARY if r['City'] != city] + [uq]
        print(f'  📊 UQ: inter-model σ={uq["Mean_sigma"]:.4f} | MC σ={uq.get("MC_Mean_sigma", float("nan")):.4f}')

    # ---- LOCO ----
    for ln in [n for n in LOCO_MODELS if n in tree_models]:
        lm = tree_models[ln]; base = roc_auc_score(yte, lm.predict_proba(Xte)[:, 1]); lrows = []
        for i, fn in enumerate(kept):
            try:
                ml = clone(lm).fit(np.delete(Xtr, i, 1), ytr)
                a = roc_auc_score(yte, ml.predict_proba(np.delete(Xte, i, 1))[:, 1])
                lrows.append({'Feature': fn, 'AUC_drop': base - a, 'AUC_without': a})
            except Exception:
                lrows.append({'Feature': fn, 'AUC_drop': np.nan, 'AUC_without': np.nan})
        ldf = pd.DataFrame(lrows).sort_values('AUC_drop', ascending=False)
        ldf['City'], ldf['Model'], ldf['Baseline_AUC'] = city, ln, base
        lp = TABLES_DIR / f'{city}_LOCO_{ln}.csv'; ldf.to_csv(lp, index=False)
        LOCO_ALL.append(ldf); CKPT.setdefault('loco_all_files', []).append(str(lp))
        print(f'  🔬 LOCO {ln}: top3={list(ldf.Feature.head(3))}')

    mdf = pd.DataFrame(metrics)
    mdf.to_csv(TABLES_DIR / f'{city}_7model_comparison.csv', index=False)
    ALL_RESULTS[city] = {'metrics': metrics, 'features_kept': kept, 'features_removed': removed,
                         'n_samples': int(len(X)), 'n_train': int(len(Xtr)), 'n_test': int(len(Xte)),
                         'n_flood': int(y.sum()), 'n_features': len(kept), 'sampling': slog,
                         'minutes': round((time.time() - t0) / 60, 1)}
    print(f'  ⏱️ {city}: {ALL_RESULTS[city]["minutes"]} min')
    cols = ['Model', 'AUC', 'PR_AUC', 'F1', 'MCC', 'Brier', 'Threshold'] + (['CV_AUC'] if 'CV_AUC' in mdf else [])
    print(mdf[cols].to_string(index=False, float_format='%.4f'))

    CKPT.update({'all_results': ALL_RESULTS, 'xai_summary': XAI_SUMMARY, 'uq_summary': UQ_SUMMARY,
                 'sampling_log': SAMPLING_LOG, 'label_checks': LABEL_CHECKS})
    CKPT['completed'].append(city)
    save_checkpoint(CKPT)
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\n✅ All cities processed')

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 5 — Summary tables
# ═════════════════════════════════════════════════════════════════════
import seaborn as sns
summ = pd.DataFrame([{'City': c, **m} for c, d in ALL_RESULTS.items() for m in d['metrics']])
summ.to_csv(TABLES_DIR / 'ALL_metrics_long.csv', index=False)

def _pivot(metric, sort=True):
    pv = summ.pivot_table(index='Model', columns='City', values=metric)
    pv['Mean'] = pv.mean(1)
    return pv.sort_values('Mean', ascending=False) if sort else pv

pivot = _pivot('AUC'); pivot.to_csv(TABLES_DIR / 'Table_AUC_Pivot.csv')
print('Spatial hold-out AUC (20 %, disjoint ~1 km blocks):\n', pivot.to_string(float_format='%.4f'))
if 'CV_AUC' in summ:
    cvp = _pivot('CV_AUC').dropna(how='all'); cvp.to_csv(TABLES_DIR / 'Table_SpatialCV_AUC_Pivot.csv')
    print('\nSpatial-block CV AUC (tree models, ~1 km blocks):\n', cvp.to_string(float_format='%.4f'))
for metric in ['PR_AUC', 'F1', 'MCC', 'Kappa', 'Brier', 'Threshold']:
    _pivot(metric, sort=False).to_csv(TABLES_DIR / f'Table_{metric}_Pivot.csv')
print('\nF1 at Youden threshold:\n', _pivot('F1', sort=False).to_string(float_format='%.3f'))

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.heatmap(pivot.drop(columns='Mean'), annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.7, vmax=1.0, linewidths=.5, ax=ax)
ax.set_title('AUC-ROC: 7 models × 5 East African cities'); plt.tight_layout()
plt.savefig(FIGURES_DIR / 'Fig_Heatmap_AUC.png', dpi=300, bbox_inches='tight'); plt.close()

if XAI_SUMMARY:
    x = pd.DataFrame(XAI_SUMMARY); x.to_csv(TABLES_DIR / 'SHAP_ALE_correlation.csv', index=False)
    print('\nSHAP↔ALE Spearman ρ:\n', x.to_string(index=False, float_format='%.3f'))
if UQ_SUMMARY:
    u = pd.DataFrame(UQ_SUMMARY); u.to_csv(TABLES_DIR / 'UQ_summary.csv', index=False)
    print('\nUQ:\n', u.to_string(index=False, float_format='%.4f'))
if LOCO_ALL:
    pd.concat(LOCO_ALL, ignore_index=True).drop_duplicates(['City', 'Model', 'Feature']).to_csv(TABLES_DIR / 'LOCO_all.csv', index=False)
if LABEL_CHECKS:
    lt = pd.DataFrame(LABEL_CHECKS); lt.to_csv(TABLES_DIR / 'Table_Label_Sanity.csv', index=False)
    print('\nLabel sanity (hydrological checks consistent / total):')
    print(lt.groupby('City')['Consistent'].agg(['sum', 'count']).to_string())
if SAMPLING_LOG:
    sl = pd.DataFrame(SAMPLING_LOG); sl.to_csv(TABLES_DIR / 'Table_Sampling_Log.csv', index=False)
    print('\nSampling log:\n', sl.to_string(index=False))

pd.DataFrame([{'City': c, 'N_samples': d['n_samples'], 'N_flood': d.get('n_flood'), 'N_train': d['n_train'],
               'N_test': d['n_test'], 'Features_kept': ', '.join(d['features_kept']),
               'Features_removed_VIF': ', '.join(d['features_removed'])}
              for c, d in ALL_RESULTS.items()]).to_csv(TABLES_DIR / 'Table_Samples_Features.csv', index=False)
pd.DataFrame([{'Predictor': k, 'Reason_excluded': v} for k, v in EXCLUDED_PREDICTORS.items()]
             ).to_csv(TABLES_DIR / 'Table_Excluded_Predictors.csv', index=False)
print(f'\n✅ Tables → {TABLES_DIR}')

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 6 — Wall-to-wall GeoTIFFs: probability, 5-class susceptibility (Jenks), uncertainty
#          clipped to city boundary, with QGIS .qml sidecar styles (auto-load in QGIS)
# ═════════════════════════════════════════════════════════════════════
from tqdm.auto import tqdm
SUSC_LABELS = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']
SUSC_COLORS = ['#1a9850', '#91cf60', '#fee08b', '#fc8d59', '#d73027']
TREE_NAMES = ['RF', 'XGBoost', 'CatBoost', 'LightGBM']

def qml_paletted(path):
    ent = ''.join(f'<paletteEntry value="{i+1}" color="{c}" label="{l}" alpha="255"/>'
                  for i, (l, c) in enumerate(zip(SUSC_LABELS, SUSC_COLORS)))
    path.write_text(f'<!DOCTYPE qgis><qgis version="3.28"><pipe><rasterrenderer type="paletted" band="1" opacity="1">'
                    f'<colorPalette>{ent}</colorPalette></rasterrenderer></pipe></qgis>')

def qml_ramp(path, vmin, vmax, colors):
    vals = np.linspace(vmin, vmax, len(colors))
    items = ''.join(f'<item value="{v:.4f}" color="{c}" label="{v:.2f}" alpha="255"/>' for v, c in zip(vals, colors))
    path.write_text(f'<!DOCTYPE qgis><qgis version="3.28"><pipe><rasterrenderer type="singlebandpseudocolor" band="1" '
                    f'opacity="1" classificationMin="{vmin}" classificationMax="{vmax}"><rastershader>'
                    f'<colorrampshader colorRampType="INTERPOLATED" classificationMode="1" clip="0">{items}'
                    f'</colorrampshader></rastershader></rasterrenderer></pipe></qgis>')

def ref_grid(cfg):
    p = find_raster(cfg['raster_dir'], REF_PREDICTOR)
    with rasterio.open(p) as r:
        h, w, tr, crs = r.height, r.width, r.transform, r.crs
    f = int(np.ceil(np.sqrt(h * w / MAX_MAP_PIXELS))) if h * w > MAX_MAP_PIXELS else 1
    if f > 1:
        tr = tr * tr.scale(f, f); h, w = h // f, w // f
        print(f'  grid decimated ×{f} for memory')
    return h, w, tr, crs

def read_on_grid(path, h, w, tr, crs, fill=False):
    a, str_, scrs = read_band(path, fill)
    dst = np.full((h, w), np.nan, np.float32)
    reproject(a, dst, src_transform=str_, src_crs=scrs, src_nodata=np.nan,
              dst_transform=tr, dst_crs=crs, dst_nodata=np.nan, resampling=Resampling.bilinear)
    return dst

def write_tif(path, arr, tr, crs, dtype, nodata):
    with rasterio.open(path, 'w', driver='GTiff', height=arr.shape[0], width=arr.shape[1], count=1,
                       dtype=dtype, crs=crs, transform=tr, nodata=nodata, compress='lzw') as d:
        d.write(arr.astype(dtype), 1)

def pixel_area_km2(tr, crs, h):
    if crs.is_geographic:
        lat = tr.f + tr.e * (np.arange(h) + 0.5)
        return (abs(tr.a) * 111.32) * (abs(tr.e) * 111.32) * np.cos(np.radians(lat))
    return np.full(h, abs(tr.a * tr.e) / 1e6)

def class_breaks(p, rng):
    if CLASS_METHOD == 'jenks':
        smp = p[rng.choice(len(p), min(20000, len(p)), replace=False)].astype(float).tolist()
        try:
            br = jenkspy.jenks_breaks(smp, n_classes=5)
        except TypeError:
            br = jenkspy.jenks_breaks(smp, nb_class=5)
        return [float(b) for b in br[1:5]]
    return [0.2, 0.4, 0.6, 0.8]

AREA_ROWS, BREAK_ROWS = [], []
rng_map = np.random.RandomState(42)
MAP_FAILED = {}
for city, cfg in CITIES.items():
 try:
     res = ALL_RESULTS.get(city)
     if res is None: print(f'⏩ {city}: no results'); continue
     out_prob = TIF_DIR / f'{city}_FloodProbability.tif'
     out_susc = TIF_DIR / f'{city}_FloodSusceptibility_5class.tif'
     out_unc  = TIF_DIR / f'{city}_Uncertainty_sigma.tif'
     out_brk  = TIF_DIR / f'{city}_class_breaks.json'
     print(f'\n🌍 {city}')

     kept = res['features_kept']
     srcs = {f: find_raster(cfg['raster_dir'], f) for f in kept}
     missing = [f for f, p in srcs.items() if p is None]
     if missing:
         print(f'  ⚠️ no raster for {missing} (CSV-only predictors) — map skipped'); continue

     h, w, tr, crs = ref_grid(cfg)
     bnd = gpd.read_file(cfg['boundary']).to_crs(crs)
     inside = ~geometry_mask(bnd.geometry, (h, w), tr)

     if out_prob.exists() and out_susc.exists() and out_unc.exists() and out_brk.exists():
         print('  ⏩ TIFs exist')
         with rasterio.open(out_susc) as s: susc = s.read(1)
         BREAK_ROWS.append(json.loads(out_brk.read_text()))
     else:
         stack = np.stack([read_on_grid(srcs[f], h, w, tr, crs, f in FILL_PREDICTORS) for f in kept])
         valid = inside & ~np.isnan(stack).any(0)
         idx = np.flatnonzero(valid.ravel())
         Xv = stack.reshape(len(kept), -1)[:, idx].T
         del stack; gc.collect()
         print(f'  {h}×{w} grid | {len(idx):,} valid px inside boundary ({len(idx)/max(inside.sum(),1):.0%} of city)')

         probs = {}
         for tn in TREE_NAMES:
             mp = MODELS_DIR / f'{city}_{tn}.pkl'
             if not mp.exists(): continue
             m = joblib.load(mp); p = np.empty(len(idx), np.float32)
             for s0 in tqdm(range(0, len(idx), 200_000), desc=tn, leave=False):
                 p[s0:s0+200_000] = m.predict_proba(Xv[s0:s0+200_000])[:, 1]
             probs[tn] = p; del m; gc.collect()

         best = max(probs, key=lambda n: next(m['AUC'] for m in res['metrics'] if m['Model'] == n))
         P = np.stack(list(probs.values()))
         def to2d(v, fill=np.nan, dt=np.float32):
             a = np.full(h * w, fill, dt); a[idx] = v; return a.reshape(h, w)
         brk = class_breaks(probs[best], rng_map)
         prob = to2d(probs[best]); unc = to2d(P.std(0))
         susc = to2d(np.clip(np.digitize(probs[best], brk) + 1, 1, 5), 0, np.uint8)
         rec = {'City': city, 'Model': best, 'Method': CLASS_METHOD, 'Breaks': [round(b, 4) for b in brk]}
         out_brk.write_text(json.dumps(rec)); BREAK_ROWS.append(rec)
         print(f'  best tree: {best} | {CLASS_METHOD} breaks: {rec["Breaks"]}')

         write_tif(out_prob, prob, tr, crs, 'float32', np.nan)
         write_tif(out_unc,  unc,  tr, crs, 'float32', np.nan)
         write_tif(out_susc, susc, tr, crs, 'uint8', 0)
         qml_ramp(out_prob.with_suffix('.qml'), 0, 1, SUSC_COLORS)
         qml_ramp(out_unc.with_suffix('.qml'), 0, float(np.nanpercentile(unc, 99)), ['#f7fbff', '#6baed6', '#08306b'])
         qml_paletted(out_susc.with_suffix('.qml'))
         print(f'  ✅ {out_prob.name}, {out_susc.name}, {out_unc.name} (+ .qml)')
         del Xv, P, probs, prob, unc; gc.collect()

     a_row = pixel_area_km2(tr, crs, h)[:, None] * np.ones((1, w))
     tot = a_row[susc > 0].sum()
     for k, lab in enumerate(SUSC_LABELS, 1):
         ak = a_row[susc == k].sum()
         AREA_ROWS.append({'City': city, 'Class': lab, 'Area_km2': round(ak, 2), 'Pct': round(100 * ak / tot, 2) if tot else 0})


 except Exception as e:
    MAP_FAILED[city] = str(e)[:200]; gc.collect()
    print(f'  ❌ {city}: map failed ({str(e)[:150]}) — other cities continue')
if BREAK_ROWS:
    pd.DataFrame(BREAK_ROWS).to_csv(TABLES_DIR / 'Table_Class_Breaks.csv', index=False)
if AREA_ROWS:
    ad = pd.DataFrame(AREA_ROWS); ad.to_csv(TABLES_DIR / 'Table_Susceptibility_Area.csv', index=False)
    print('\n', ad.pivot(index='Class', columns='City', values='Pct').reindex(SUSC_LABELS).to_string())
print(f'\n✅ GeoTIFFs → {TIF_DIR}  (drag into QGIS: styles load automatically)')
if MAP_FAILED:
    print('❌ Maps NOT produced for:', MAP_FAILED, '\n   → run this cell again (finished cities are skipped).')

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 7 — Cartographic maps (north arrow, scale bar, graticule, legend), 300 dpi
# ═════════════════════════════════════════════════════════════════════
from matplotlib.colors import ListedColormap, BoundaryNorm, LinearSegmentedColormap
from matplotlib.patches import Patch

CMAP_SUSC = ListedColormap(SUSC_COLORS); NORM_SUSC = BoundaryNorm(np.arange(0.5, 6.5), 5)
CMAP_PROB = LinearSegmentedColormap.from_list('prob', SUSC_COLORS)

def add_north_arrow(ax, x=0.94, y=0.90):
    ax.annotate('N', xy=(x, y), xytext=(x, y - 0.09), xycoords='axes fraction', ha='center', va='center',
                fontsize=12, fontweight='bold', arrowprops=dict(facecolor='black', width=4, headwidth=11))

def add_scalebar(ax, crs, extent):
    x0, x1, y0, y1 = extent
    km_per_unit = 111.32 * np.cos(np.radians((y0 + y1) / 2)) if crs.is_geographic else 1e-3
    span_km = (x1 - x0) * km_per_unit
    nice = [1, 2, 5, 10, 20, 50]; L = max(n for n in nice if n <= span_km / 4) if span_km >= 4 else 1
    Lu = L / km_per_unit
    bx, by = x0 + 0.05 * (x1 - x0), y0 + 0.06 * (y1 - y0)
    ax.add_patch(plt.Rectangle((bx, by), Lu / 2, (y1 - y0) * 0.012, fc='black', ec='black'))
    ax.add_patch(plt.Rectangle((bx + Lu / 2, by), Lu / 2, (y1 - y0) * 0.012, fc='white', ec='black'))
    for frac, txt in [(0, '0'), (0.5, f'{L/2:g}'), (1, f'{L:g} km')]:
        ax.text(bx + frac * Lu, by + (y1 - y0) * 0.025, txt, ha='center', fontsize=8)

def read_tif(p):
    with rasterio.open(p) as s:
        a = s.read(1).astype(float); b = s.bounds
        if s.nodata is not None and not np.isnan(s.nodata): a[a == s.nodata] = np.nan
        return a, (b.left, b.right, b.bottom, b.top), s.crs

composite = []
for city, cfg in CITIES.items():
 try:
     ps = TIF_DIR / f'{city}_FloodSusceptibility_5class.tif'
     pp = TIF_DIR / f'{city}_FloodProbability.tif'
     pu = TIF_DIR / f'{city}_Uncertainty_sigma.tif'
     if not ps.exists(): continue
     susc, ext, crs = read_tif(ps); susc[susc == 0] = np.nan
     prob, _, _ = read_tif(pp); unc, _, _ = read_tif(pu)
     bnd = gpd.read_file(cfg['boundary']).to_crs(crs)
     composite.append((city, susc, ext, crs, bnd))

     fig, axes = plt.subplots(1, 3, figsize=(20, 7))
     for ax, arr, cmap, norm, title in [
         (axes[0], prob, CMAP_PROB, None, 'Flood probability'),
         (axes[1], susc, CMAP_SUSC, NORM_SUSC, 'Flood susceptibility (5 classes)'),
         (axes[2], unc, 'Blues', None, 'Uncertainty (inter-model σ)')]:
         im = ax.imshow(arr, extent=ext, cmap=cmap, norm=norm, interpolation='nearest',
                        vmin=None if norm else 0, vmax=None if norm else (1 if 'prob' in title else np.nanpercentile(arr, 99)))
         bnd.boundary.plot(ax=ax, color='black', linewidth=0.8)
         ax.set_title(title, fontsize=13, fontweight='bold')
         ax.set_xlabel('Longitude (°E)' if crs.is_geographic else 'Easting (m)')
         ax.set_ylabel('Latitude (°)' if crs.is_geographic else 'Northing (m)')
         ax.grid(ls=':', lw=0.4, color='grey'); ax.tick_params(labelsize=8)
         add_north_arrow(ax); add_scalebar(ax, crs, ext)
         if norm is None:
             fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
         else:
             ax.legend(handles=[Patch(color=c, label=l) for c, l in zip(SUSC_COLORS, SUSC_LABELS)],
                       loc='lower right', fontsize=8, title='Class', title_fontsize=9, framealpha=0.9)
     fig.suptitle(f'{city.replace("_", " ")} — flood exposure mapping ({crs.to_string()})', fontsize=15, y=1.01)
     plt.tight_layout()
     plt.savefig(MAPS_DIR / f'Map_{city}.png', dpi=300, bbox_inches='tight'); plt.close()
     print(f'✅ Map_{city}.png')


 except Exception as e:
    plt.close('all'); print(f'❌ Map_{city}.png failed: {str(e)[:150]}')
if composite:
    fig, axes = plt.subplots(1, len(composite), figsize=(5 * len(composite), 6))
    axes = np.atleast_1d(axes)
    for ax, (city, susc, ext, crs, bnd) in zip(axes, composite):
        ax.imshow(susc, extent=ext, cmap=CMAP_SUSC, norm=NORM_SUSC, interpolation='nearest')
        bnd.boundary.plot(ax=ax, color='black', linewidth=0.7)
        ax.set_title(city.replace('_', ' '), fontsize=13, fontweight='bold')
        ax.tick_params(labelsize=7); ax.grid(ls=':', lw=0.3, color='grey')
        add_north_arrow(ax); add_scalebar(ax, crs, ext)
    fig.legend(handles=[Patch(color=c, label=l) for c, l in zip(SUSC_COLORS, SUSC_LABELS)],
               loc='lower center', ncol=5, fontsize=10, bbox_to_anchor=(0.5, -0.04))
    plt.tight_layout()
    plt.savefig(MAPS_DIR / 'Map_Composite_5cities_Susceptibility.png', dpi=300, bbox_inches='tight'); plt.close()
    print('✅ Map_Composite_5cities_Susceptibility.png')

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 8 — Publication figures: ROC curves, AUC bars, LOCO heatmap, UQ
# ═════════════════════════════════════════════════════════════════════
import seaborn as sns
MODEL_ORDER = ['RF', 'XGBoost', 'CatBoost', 'LightGBM', 'ResNet1D', 'TabTransformer', 'Stacking']
MCOL = dict(zip(MODEL_ORDER, sns.color_palette('tab10', 7)))

cities_done = [c for c in CITIES if (ROC_DIR / f'{c}.npz').exists()]
fig, axes = plt.subplots(1, len(cities_done), figsize=(5 * len(cities_done), 5), squeeze=False)
for ax, city in zip(axes[0], cities_done):
    d = np.load(ROC_DIR / f'{city}.npz'); yt = d['y_true']
    for m in MODEL_ORDER:
        if m in d.files:
            fpr, tpr, _ = roc_curve(yt, d[m])
            ax.plot(fpr, tpr, color=MCOL[m], lw=1.4, label=f'{m} ({roc_auc_score(yt, d[m]):.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.5)
    ax.set_title(city.replace('_', ' '), fontweight='bold'); ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
    ax.legend(fontsize=7, loc='lower right'); ax.set_aspect('equal')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'Fig_ROC_AllCities.png', dpi=300, bbox_inches='tight'); plt.close()
print('✅ Fig_ROC_AllCities.png')

bp = summ.pivot(index='Model', columns='City', values='AUC').reindex([m for m in MODEL_ORDER if m in summ.Model.unique()])
ax = bp.plot(kind='bar', figsize=(13, 5.5), width=0.8)
ax.axhline(0.85, color='red', ls='--', lw=1, label='AUC = 0.85')
ax.set_ylim(0.6, 1.0); ax.set_ylabel('AUC-ROC'); ax.set_title('Model performance across five East African cities')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left'); plt.xticks(rotation=30, ha='right'); plt.tight_layout()
plt.savefig(FIGURES_DIR / 'Fig_BarChart_AUC.png', dpi=300, bbox_inches='tight'); plt.close()
print('✅ Fig_BarChart_AUC.png')

if LOCO_ALL:
    L = pd.concat(LOCO_ALL).drop_duplicates(['City', 'Model', 'Feature'])
    lp = L.groupby(['Feature', 'City'])['AUC_drop'].mean().unstack()
    top = lp.loc[lp.mean(1).nlargest(10).index]
    plt.figure(figsize=(10, 6))
    sns.heatmap(top, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=.5)
    plt.title('LOCO AUC drop — top 10 predictors (mean of RF, XGBoost, CatBoost)'); plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'Fig_LOCO_Heatmap.png', dpi=300, bbox_inches='tight'); plt.close()
    print('✅ Fig_LOCO_Heatmap.png')

if UQ_SUMMARY:
    u = pd.DataFrame(UQ_SUMMARY)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
    a1.bar(u.City, u.Mean_sigma, color='steelblue'); a1.set_title('Inter-model σ (4 tree models)'); a1.set_ylabel('Mean σ')
    if 'MC_Mean_sigma' in u:
        a2.bar(u.City, u.MC_Mean_sigma, color='coral'); a2.set_title(f'MC Dropout σ (ResNet1D, T={MC_DROPOUT_T})'); a2.set_ylabel('Mean σ')
    for a in (a1, a2): a.tick_params(axis='x', rotation=20)
    plt.tight_layout(); plt.savefig(FIGURES_DIR / 'Fig_UQ_Bars.png', dpi=300, bbox_inches='tight'); plt.close()
    print('✅ Fig_UQ_Bars.png')

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 9 — External validation vs Global Flood Database v1 (Tellman et al., 2021, Nature 596:80–86)
#   913 MODIS-mapped flood events 2000–2018, 250 m. Independent of our inventories.
#   Positive = flooded in ≥1 GFD event; Negative = never flooded; JRC permanent water excluded.
#   If the login popup appears, choose the Google account that owns GEE_PROJECT.
# ═════════════════════════════════════════════════════════════════════
import requests, zipfile, io, shutil
try:
    import ee
except ImportError:
    ee = None
EXTVAL_NOT_APPLICABLE = {
    'Kampala': 'inventory = urban pluvial (drainage-driven) flooding; GFD records fluvial/lacustrine floods at 250 m',
}
# GFD tiles depend only on the city boundary, so tiles cached by an earlier run are reused (no GEE needed).
for _old in ['M2_FINAL_v3_Results', 'M2_FINAL_v2_Results']:
    for _f in (BASE / _old / 'External_Validation').glob('*_GFD_freq_perm.tif'):
        if not (EXTVAL_DIR / _f.name).exists() and _f.stat().st_size > 500:
            shutil.copyfile(_f, EXTVAL_DIR / _f.name); print(f'  ↪ reused cached {_f.name} ({_old})')

GEE_OK = True
try:
    try:
        ee.Initialize(project=GEE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT)
    gfd = ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1')
    gfd_img = (gfd.select('flooded').sum().rename('freq')
               .addBands(gfd.select('jrc_perm_water').max().rename('perm'))
               .unmask(0).toFloat())
except Exception as e:
    GEE_OK = False
    print(f'⚠️ Google Earth Engine unavailable ({str(e)[:120]}). Cached GFD tiles are used where available.')

EXT_ROWS = []
rng = np.random.RandomState(42)
for city, cfg in CITIES.items():
  try:
    pp = TIF_DIR / f'{city}_FloodProbability.tif'
    if not pp.exists():
        print(f'⏩ {city}: no FloodProbability GeoTIFF (run Cell 6)'); continue
    gp = EXTVAL_DIR / f'{city}_GFD_freq_perm.tif'
    n_events = np.nan
    if GEE_OK:
        b = gpd.read_file(cfg['boundary']).to_crs('EPSG:4326').total_bounds
        roi = ee.Geometry.Rectangle([float(v) for v in b])
        n_events = gfd.filterBounds(roi).filter(ee.Filter.gt('dfo_severity', 0)).size().getInfo()
        if not gp.exists():
            r = requests.get(gfd_img.clip(roi).getDownloadURL({'scale': 250, 'region': roi, 'crs': 'EPSG:4326',
                                                               'format': 'GEO_TIFF'}))
            content = r.content
            if content[:2] == b'PK':
                with zipfile.ZipFile(io.BytesIO(content)) as z:
                    content = z.read([n for n in z.namelist() if n.endswith('.tif')][0])
            gp.write_bytes(content)
    if not gp.exists():
        print(f'⏩ {city}: no GFD tile and no GEE access — skipped'); continue

    prob, tr, crs, _ = safe_read(pp); h, w = prob.shape
    freq = np.full((h, w), np.nan, np.float32); perm = np.full((h, w), np.nan, np.float32)
    with rasterio.open(gp) as s:
        for band, dst in [(1, freq), (2, perm)]:
            reproject(s.read(band).astype(np.float32), dst, src_transform=s.transform, src_crs=s.crs,
                      dst_transform=tr, dst_crs=crs, resampling=Resampling.nearest)
    ok = ~np.isnan(prob) & ~np.isnan(freq) & (perm == 0)
    pos = np.flatnonzero((ok & (freq >= 1)).ravel())
    neg = np.flatnonzero((ok & (freq == 0)).ravel())
    row = {'City': city, 'GFD_events_in_bbox': n_events, 'N_flooded_px': int(len(pos)), 'N_dry_px': int(len(neg)),
           'Applicable': city not in EXTVAL_NOT_APPLICABLE, 'Reason': EXTVAL_NOT_APPLICABLE.get(city, '')}
    if len(pos) < 500:
        row['Note'] = 'too few GFD flood pixels for a robust AUC'
        print(f'⚠️ {city}: {len(pos)} GFD-flooded px — AUC not reported')
    else:
        n = min(20000, len(pos), len(neg))
        ps, ns = rng.choice(pos, n, replace=False), rng.choice(neg, n, replace=False)
        yv = np.r_[np.ones(n), np.zeros(n)]; pv = prob.ravel()[np.r_[ps, ns]]
        row.update({'N_sampled_each': int(n), 'AUC_vs_GFD': float(roc_auc_score(yv, pv)),
                    'Mean_prob_GFD_flooded': float(pv[:n].mean()), 'Mean_prob_GFD_dry': float(pv[n:].mean())})
        print(f'✅ {city}: AUC vs GFD = {row["AUC_vs_GFD"]:.3f} ({len(pos):,} flooded px)'
              + ('' if row['Applicable'] else '  [reported; flagged not applicable: pluvial inventory]'))
    EXT_ROWS.append(row)
    del prob, freq, perm; gc.collect()
  except Exception as e:
    print(f'❌ {city}: external validation failed ({str(e)[:150]})')

if EXT_ROWS:
    ev = pd.DataFrame(EXT_ROWS); ev.to_csv(TABLES_DIR / 'Table_External_Validation_GFD.csv', index=False)
    print('\n', ev.to_string(index=False, float_format='%.3f'))


In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 10b — Informal-settlement (slum) probability maps (manuscript Sections 2.2.1, 2.4.5; Table S3; Figure 2)
#   Inputs (per city, folder slum_flood_detection_East_African_city):
#     • Sentinel-2 Level-2A band exports for 2025 and 2020 (Google Earth Engine GeoTIFFs; split exports are mosaicked)
#     • 2025 reference samples: class (1 = slum, 0 = non-slum), longitude, latitude
#   Steps:
#     1. Predictor values (B2, B3, B4, B8, B11, B12) are read from the 2025 export at every sample point
#     2. Accuracy: samples grouped into ≈1 km blocks (0.01°); 30 % of blocks held out; 100-tree random forest → Table S3
#     3. Final 100-tree random forest trained on all samples; applied to the 2025 and 2020 exports inside the city
#        boundary → slum probability = share of trees voting "slum" (Eq. 11), written as GeoTIFF
#   Outputs: Slum_Maps/<city>_<year>_SlumProb.tif, Supplementary/Table_S3_Slum_Accuracy_SpatialBlocks.csv,
#            Slum_Maps/Slum_Feature_Check.csv, Figures/Fig2_Slum_Probability_2020_2025.png
# ═════════════════════════════════════════════════════════════════════
import glob, json as _json
from rasterio.features import geometry_mask
from rasterio.windows import Window
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score,
                             matthews_corrcoef, balanced_accuracy_score, roc_auc_score, jaccard_score)

SLUM_BASE = BASE / 'slum_flood_detection_East_African_city'
SLUM_OUT = RESULTS_DIR / 'Slum_Maps'; SLUM_OUT.mkdir(parents=True, exist_ok=True)
SUPP_DIR = RESULTS_DIR / 'Supplementary'; SUPP_DIR.mkdir(parents=True, exist_ok=True)
S2_BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
RF_TREES, BLOCK_DEG, TEST_FRAC, ROW_BLOCK, SEED = 100, 0.01, 0.30, 256, 42
MAKE_2020 = True                                   # set False to map 2025 only (faster)

# Sentinel-2 exports and reference samples per city (file names exactly as in Drive)
SLUM_INPUTS = {
    'Kigali':        dict(dir='Kigali',        samples='*Training_Samples*2025.csv',
                          s2={'2025': ['Sentinel2_Raw_Bands.tif'], '2020': ['Sentinel2_Raw_Bands-2020.tif']}),
    'Kampala':       dict(dir='kampala',       samples='*Training_Samples*2025.csv',
                          s2={'2025': ['Sentinel2_Raw_Bands-2025.tif'], '2020': ['Sentinel2_Raw_Bands-2020.tif']}),
    'Nairobi':       dict(dir='Nairobi',       samples='*Training_Samples*2025.csv',
                          s2={'2025': ['Sentinel2_Raw_Bands-2025.tif'], '2020': ['Sentinel2_Raw_Bands-2020.tif']}),
    'Dar_es_Salaam': dict(dir='Dar es Salaam', samples='*Training_Samples*2025.csv',
                          s2={'2025': ['_Sentinel2_Raw_Bands-2025.tif', '_Sentinel2_Raw_Bands-0000000000-0000009472.tif'],
                              '2020': ['Sentinel2_Raw_Bands-2020-0000000000-0000000000.tif',
                                       'Sentinel2_Raw_Bands-2020-0000000000-0000009472.tif']}),
    'Bujumbura':     dict(dir='Bujumbura',     samples='*Training_Samples*2025.csv',
                          s2={'2025': ['Bujumbura_Sentinel2_Raw_2025.tif'], '2020': ['Sentinel2_Raw_Bands2020.tif']}),
}


class S2Mosaic:
    """Read-only mosaic of one or more aligned GEE export tiles; returns the six bands as float32 (NaN = no data)."""
    def __init__(self, paths):
        self.srcs = [rasterio.open(p) for p in paths]
        s0 = self.srcs[0]
        self.res, self.crs = s0.res, s0.crs
        for s in self.srcs[1:]:
            assert np.allclose(s.res, self.res) and s.crs == self.crs, 'tiles differ in resolution/CRS'
        self.band_idx = self._band_index(s0)
        x0 = min(s.bounds.left for s in self.srcs); y0 = max(s.bounds.top for s in self.srcs)
        x1 = max(s.bounds.right for s in self.srcs); y1 = min(s.bounds.bottom for s in self.srcs)
        self.transform = rasterio.Affine(self.res[0], 0, x0, 0, -self.res[1], y0)
        self.width = int(round((x1 - x0) / self.res[0])); self.height = int(round((y0 - y1) / self.res[1]))
        self.off = [(int(round((y0 - s.bounds.top) / self.res[1])), int(round((s.bounds.left - x0) / self.res[0])))
                    for s in self.srcs]

    @staticmethod
    def _band_index(s):
        desc = [(d or '').strip().upper() for d in s.descriptions]
        if all(b in desc for b in S2_BANDS):
            return [desc.index(b) + 1 for b in S2_BANDS]
        if s.count == len(S2_BANDS):
            return list(range(1, 7))            # no band names stored: export order B2, B3, B4, B8, B11, B12
        raise ValueError(f'{s.name}: {s.count} bands without names {s.descriptions} — cannot identify B2…B12')

    def read_rows(self, r0, r1):
        out = np.full((len(S2_BANDS), r1 - r0, self.width), np.nan, np.float32)
        for s, (ro, co) in zip(self.srcs, self.off):
            a0, a1 = max(r0, ro), min(r1, ro + s.height)
            if a0 >= a1: continue
            win = Window(0, a0 - ro, s.width, a1 - a0)
            arr = s.read(self.band_idx, window=win).astype(np.float32)
            if s.nodata is not None: arr[arr == s.nodata] = np.nan
            out[:, a0 - r0:a1 - r0, co:co + s.width] = arr
        out[:, np.all(out == 0, axis=0)] = np.nan            # GEE masked pixels are exported as 0
        return out

    def sample(self, lon, lat):
        v = np.full((len(lon), len(S2_BANDS)), np.nan, np.float32)
        for s, (ro, co) in zip(self.srcs, self.off):
            rr, cc = rasterio.transform.rowcol(s.transform, lon, lat)
            rr, cc = np.asarray(rr), np.asarray(cc)
            ok = (rr >= 0) & (rr < s.height) & (cc >= 0) & (cc < s.width)
            if not ok.any(): continue
            idx = np.flatnonzero(ok)
            for b0 in range(int(rr[idx].min()) // ROW_BLOCK * ROW_BLOCK, int(rr[idx].max()) + 1, ROW_BLOCK):
                sel = idx[(rr[idx] >= b0) & (rr[idx] < b0 + ROW_BLOCK)]
                if not len(sel): continue
                b1 = min(s.height, b0 + ROW_BLOCK)
                arr = s.read(self.band_idx, window=Window(0, b0, s.width, b1 - b0)).astype(np.float32)
                v[sel] = arr[:, rr[sel] - b0, cc[sel]].T
            if s.nodata is not None: v[ok & np.any(v == s.nodata, axis=1)] = np.nan
        v[np.all(v == 0, axis=1)] = np.nan
        return v

    def close(self):
        for s in self.srcs: s.close()


def _s2_paths(city, year):
    inp = SLUM_INPUTS[city]; ps = [SLUM_BASE / inp['dir'] / f for f in inp['s2'][year]]
    missing = [p.name for p in ps if not p.exists()]
    if missing: raise FileNotFoundError(f'{city} {year}: missing {missing}')
    return ps


SLUM_ACC, FEAT_CHECK, SLUM_PROB_NEW = [], [], {}
for city, cfg in CITIES.items():
    if city not in SLUM_INPUTS: continue
    t0 = time.time()
    try:
        inp = SLUM_INPUTS[city]
        f = sorted(glob.glob(str(SLUM_BASE / inp['dir'] / inp['samples'])))
        if not f: raise FileNotFoundError('no reference-sample CSV')
        smp = pd.read_csv(f[0]).dropna(subset=['class', 'longitude', 'latitude'])
        y = smp['class'].astype(int).values

        # 1) predictor values from the 2025 export at the sample points
        m25 = S2Mosaic(_s2_paths(city, '2025'))
        X = m25.sample(smp['longitude'].values, smp['latitude'].values)
        ok = np.isfinite(X).all(1)
        if all(b in smp.columns for b in S2_BANDS):            # provenance check against values stored in the CSV
            r = [np.corrcoef(X[ok, i], smp.loc[ok, b].values)[0, 1] for i, b in enumerate(S2_BANDS)]
            FEAT_CHECK.append({'City': city, **{f'r_{b}': round(v, 4) for b, v in zip(S2_BANDS, r)}})
        X, y, lon, lat = X[ok], y[ok], smp['longitude'].values[ok], smp['latitude'].values[ok]
        print(f'\n🏘️  {city}: {len(y):,} samples ({100 * y.mean():.1f}% slum) | '
              f'{m25.width}×{m25.height} px, {len(m25.srcs)} tile(s)')

        # 2) accuracy on held-out spatial blocks
        g = np.floor(lon / BLOCK_DEG).astype(int).astype(str) + '_' + np.floor(lat / BLOCK_DEG).astype(int).astype(str)
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=SEED).split(X, y, g))
        rf = RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=SEED).fit(X[tr], y[tr])
        pr = rf.predict_proba(X[te])[:, 1]; yp = (pr >= 0.5).astype(int); yt = y[te]
        acc = {'City': city, 'N_samples': len(y), 'Slum_share_%': round(100 * y.mean(), 1),
               'N_blocks': len(np.unique(g)), 'N_train': len(tr), 'N_test': len(te),
               'Accuracy': accuracy_score(yt, yp), 'Precision': precision_score(yt, yp, zero_division=0),
               'Recall': recall_score(yt, yp, zero_division=0), 'F1': f1_score(yt, yp, zero_division=0),
               'IoU': jaccard_score(yt, yp, zero_division=0), 'Kappa': cohen_kappa_score(yt, yp),
               'MCC': matthews_corrcoef(yt, yp), 'Balanced_Acc': balanced_accuracy_score(yt, yp),
               'ROC_AUC': roc_auc_score(yt, pr) if len(set(yt)) == 2 else np.nan}
        SLUM_ACC.append(acc)
        print(f'   spatial-block test: F1 = {acc["F1"]:.3f}, κ = {acc["Kappa"]:.3f}, balanced acc = {acc["Balanced_Acc"]:.3f}')

        # 3) final model on all samples → maps
        rf = RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=SEED).fit(X, y)
        bnd = gpd.read_file(cfg['boundary']).to_crs(m25.crs)
        for year in (['2025', '2020'] if MAKE_2020 else ['2025']):
            out = SLUM_OUT / f'{city}_{year}_SlumProb.tif'
            if out.exists():
                print(f'   ⏩ {out.name} exists'); SLUM_PROB_NEW.setdefault(city, {})[year] = out; continue
            m = m25 if year == '2025' else S2Mosaic(_s2_paths(city, year))
            inside = ~geometry_mask(bnd.geometry, out_shape=(m.height, m.width), transform=m.transform)
            prof = dict(driver='GTiff', height=m.height, width=m.width, count=1, dtype='float32', crs=m.crs,
                        transform=m.transform, nodata=np.nan, compress='deflate', tiled=True,
                        blockxsize=256, blockysize=256, BIGTIFF='IF_SAFER')
            tmp = out.with_suffix('.part.tif')
            with rasterio.open(tmp, 'w', **prof) as dst:
                for r0 in range(0, m.height, ROW_BLOCK):
                    r1 = min(m.height, r0 + ROW_BLOCK)
                    blk = m.read_rows(r0, r1)
                    v = inside[r0:r1] & np.isfinite(blk).all(0)
                    p = np.full((r1 - r0, m.width), np.nan, np.float32)
                    if v.any():
                        p[v] = rf.predict_proba(blk[:, v].T)[:, 1]
                    dst.write(p[None], window=Window(0, r0, m.width, r1 - r0))
                dst.update_tags(method=f'RandomForest n_estimators={RF_TREES} seed={SEED}; trained on 2025 samples',
                                bands=','.join(S2_BANDS), year=year)
            os.replace(tmp, out)
            SLUM_PROB_NEW.setdefault(city, {})[year] = out
            if m is not m25: m.close()
            print(f'   ✅ {out.name}')
        m25.close(); del X, rf; gc.collect()
        print(f'   {city} done in {(time.time() - t0) / 60:.1f} min')
    except Exception as e:
        print(f'❌ {city}: slum mapping failed ({str(e)[:200]}) — other cities continue')

if SLUM_ACC:
    s3 = pd.DataFrame(SLUM_ACC).round(3)
    s3.to_csv(SUPP_DIR / 'Table_S3_Slum_Accuracy_SpatialBlocks.csv', index=False)
    print('\nTable S3\n', s3.to_string(index=False))
if FEAT_CHECK:
    fc = pd.DataFrame(FEAT_CHECK); fc.to_csv(SLUM_OUT / 'Slum_Feature_Check.csv', index=False)
    print('\nCorrelation between band values read from the exports and those stored in the sample CSVs '
          '(≈1 = samples were drawn from these exports)\n', fc.to_string(index=False))

# Figure 2 — slum probability 2020, 2025 and change
cities_fig = [c for c in CITIES if c in SLUM_PROB_NEW and '2025' in SLUM_PROB_NEW[c]]
if cities_fig:
    def _ds(p, maxpx=900):
        with rasterio.open(p) as s:
            k = max(1, int(np.ceil(max(s.height, s.width) / maxpx)))
            return s.read(1, out_shape=(s.height // k, s.width // k), resampling=Resampling.average).astype(np.float32)
    ncol = 3 if MAKE_2020 else 1
    fig, axs = plt.subplots(len(cities_fig), ncol, figsize=(4.2 * ncol, 3.6 * len(cities_fig)), squeeze=False)
    for i, c in enumerate(cities_fig):
        p25 = _ds(SLUM_PROB_NEW[c]['2025'])
        cols = [('2025', p25, 'magma_r', (0, 1))]
        if MAKE_2020 and '2020' in SLUM_PROB_NEW[c]:
            p20 = _ds(SLUM_PROB_NEW[c]['2020'])
            h_, w_ = min(p20.shape[0], p25.shape[0]), min(p20.shape[1], p25.shape[1])
            cols = [('2020', p20, 'magma_r', (0, 1)), ('2025', p25, 'magma_r', (0, 1)),
                    ('Change 2020–2025', p25[:h_, :w_] - p20[:h_, :w_], 'RdBu_r', (-0.5, 0.5))]
        for j, (ttl, a, cm, (vmin, vmax)) in enumerate(cols):
            ax = axs[i, j]; im = ax.imshow(np.ma.masked_invalid(a), cmap=cm, vmin=vmin, vmax=vmax)
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f'({chr(97 + i * ncol + j)}) {c.replace("_", " ")} — {ttl}', fontsize=9)
            plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02,
                         label='Δ probability' if 'Change' in ttl else 'Slum probability')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'Fig2_Slum_Probability_2020_2025.png', dpi=300, bbox_inches='tight'); plt.close()
    print(f'\n✅ Figure 2 → {FIGURES_DIR / "Fig2_Slum_Probability_2020_2025.png"}')


In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 11 — Informal-settlement flood exposure (manuscript Section 2.6; Tables 1–2)
#   Overlays the 5-class flood-susceptibility GeoTIFF from Cell 6 on each city's 10 m slum-probability raster:
#     Table 1: very-high slum probability (p > 0.8) area, very-high flood susceptibility (class 5) area,
#              and their overlap, in hectares
#     Table 2: chi-square test of independence on the 2×2 table [slum p > 0.8] × [very-high class],
#              and the Relative Flood Exposure Index RFEI = P(very high | slum) / P(very high | non-slum)
# ═════════════════════════════════════════════════════════════════════
from scipy.stats import chi2_contingency

SLUM_THRESHOLD = 0.8
# Slum-probability GeoTIFFs (2025) produced by Cell S from the Sentinel-2 exports (Section 2.4.5)
SLUM_PROB = {c: v['2025'] for c, v in SLUM_PROB_NEW.items() if '2025' in v} if 'SLUM_PROB_NEW' in globals() else {}
for c in CITIES:
    SLUM_PROB.setdefault(c, RESULTS_DIR / 'Slum_Maps' / f'{c}_2025_SlumProb.tif')
CHI2_SAMPLE_N = None   # None = test on all valid pixels; an integer = test on a random pixel sample of that size

def _pixel_area_km2(tr, crs, h):
    if crs.is_geographic:
        lat = tr.f + tr.e * (np.arange(h) + 0.5)
        return (abs(tr.a) * 111.32) * (abs(tr.e) * 111.32) * np.cos(np.radians(lat))
    return np.full(h, abs(tr.a * tr.e) / 1e6)

def exposure_one_city(city):
    susc_p, slum_p = TIF_DIR / f'{city}_FloodSusceptibility_5class.tif', SLUM_PROB.get(city)
    if not susc_p.exists():
        print(f'⏩ {city}: run Cell 6 first (no susceptibility GeoTIFF)'); return None, None
    if slum_p is None or not Path(slum_p).exists():
        print(f'⏩ {city}: slum-probability raster not found → set SLUM_PROB[{city!r}]'); return None, None
    # Overlay on the native 10 m slum grid: flood classes are resampled (nearest) onto it, so slum
    # probabilities are never smoothed and the p > 0.8 threshold keeps its meaning.
    slum, tr, crs, _ = safe_read(slum_p)
    h, w = slum.shape
    cls = np.zeros((h, w), np.int16)
    with rasterio.open(susc_p) as s:
        nd = s.nodata
        reproject(s.read(1).astype(np.int16), cls, src_transform=s.transform, src_crs=s.crs,
                  src_nodata=nd if nd is not None else 0, dst_transform=tr, dst_crs=crs, dst_nodata=0,
                  resampling=Resampling.nearest)
    nd = 0
    valid = np.isfinite(slum) & (cls >= 1) & (cls <= 5)
    if nd is not None:
        valid &= cls != nd
    row_ha = _pixel_area_km2(tr, crs, h) * 100.0                 # hectares per pixel, by row (memory-light)
    area = lambda m: round(float((m.sum(1) * row_ha).sum()), 2)
    s_bin, v_bin = (slum > SLUM_THRESHOLD) & valid, (cls == 5) & valid
    expo = {'City': city,
            'VeryHigh_Slum_Probability_ha': area(s_bin),
            'VeryHigh_Flood_Susceptibility_ha': area(v_bin),
            'Overlap_ha': area(s_bin & v_bin),
            'Valid_area_ha': area(valid)}
    sv, vv = s_bin[valid], v_bin[valid]
    if CHI2_SAMPLE_N and CHI2_SAMPLE_N < sv.size:
        idx = np.random.RandomState(42).choice(sv.size, CHI2_SAMPLE_N, replace=False); sv, vv = sv[idx], vv[idx]
    n11, n10 = int(np.sum(sv & vv)), int(np.sum(sv & ~vv))
    n01, n00 = int(np.sum(~sv & vv)), int(np.sum(~sv & ~vv))
    if min(n11 + n10, n01 + n00, n11 + n01, n10 + n00) == 0:
        print(f'  ⚠️ {city}: empty row/column in contingency table — chi-square skipped'); return expo, None
    chi2, p, _, _ = chi2_contingency([[n11, n10], [n01, n00]], correction=False)
    rfei = (n11 / (n11 + n10)) / (n01 / (n01 + n00)) if n01 > 0 else np.nan
    phi = float(np.sqrt(chi2 / sv.size))          # effect size: with millions of pixels p is always tiny
    chi = {'City': city, 'n_pixels': int(sv.size), 'slum&VH': n11, 'slum&notVH': n10,
           'nonslum&VH': n01, 'nonslum&notVH': n00, 'Chi2': round(float(chi2), 2),
           'p_value': float(p), 'Phi': round(phi, 4), 'RFEI': round(float(rfei), 2)}
    print(f'✅ {city}: overlap {expo["Overlap_ha"]} ha | χ² = {chi2:.1f}, p = {p:.3g}, φ = {phi:.3f} | RFEI = {rfei:.2f}')
    return expo, chi

EXPO_ROWS, CHI_ROWS = [], []
for city in CITIES:
    try:
        e_, c_ = exposure_one_city(city)
        if e_: EXPO_ROWS.append(e_)
        if c_: CHI_ROWS.append(c_)
    except Exception as e:
        print(f'❌ {city}: exposure failed ({str(e)[:150]})')
    gc.collect()

if EXPO_ROWS:
    pd.DataFrame(EXPO_ROWS).to_csv(TABLES_DIR / 'Table1_Exposure_Areas.csv', index=False)
    pd.DataFrame(CHI_ROWS).to_csv(TABLES_DIR / 'Table2_Chi2_RFEI.csv', index=False)
    print('\nTable 1\n', pd.DataFrame(EXPO_ROWS).to_string(index=False))
    print('\nTable 2\n', pd.DataFrame(CHI_ROWS).to_string(index=False))


In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 11b — Exposure and slum-accuracy figures (manuscript Figure 4 and Figure 6a)
#   Figure 4: per city — 2025 slum probability | 5-class flood susceptibility | exposure hotspots
#             (slum probability > 0.8 AND very-high susceptibility), all on the 10 m slum grid
#   Figure 6a: slum-classifier accuracy on spatially independent blocks (Table S3)
# ═════════════════════════════════════════════════════════════════════
from matplotlib.colors import ListedColormap, BoundaryNorm

def _on_slum_grid(city, maxpx=900):
    sp, cp = SLUM_PROB.get(city), TIF_DIR / f'{city}_FloodSusceptibility_5class.tif'
    if sp is None or not Path(sp).exists() or not cp.exists(): return None
    with rasterio.open(sp) as s:
        k = max(1, int(np.ceil(max(s.height, s.width) / maxpx)))
        h, w = s.height // k, s.width // k
        slum = s.read(1, out_shape=(h, w), resampling=Resampling.average).astype(np.float32)
        tr = s.transform * s.transform.scale(s.width / w, s.height / h); crs = s.crs
    cls = np.zeros((h, w), np.int16)
    with rasterio.open(cp) as c:
        reproject(c.read(1).astype(np.int16), cls, src_transform=c.transform, src_crs=c.crs,
                  src_nodata=c.nodata if c.nodata is not None else 0, dst_transform=tr, dst_crs=crs,
                  dst_nodata=0, resampling=Resampling.nearest)
    return slum, cls

CLS_CMAP = ListedColormap(['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c'])
rows = [(c, _on_slum_grid(c)) for c in CITIES]
rows = [(c, r) for c, r in rows if r is not None]
if rows:
    fig, axs = plt.subplots(len(rows), 3, figsize=(12.5, 3.6 * len(rows)), squeeze=False)
    for i, (c, (slum, cls)) in enumerate(rows):
        valid = np.isfinite(slum) & (cls >= 1)
        hot = np.where(valid, ((slum > SLUM_THRESHOLD) & (cls == 5)).astype(float), np.nan)
        panels = [(np.ma.masked_invalid(np.where(valid, slum, np.nan)), 'magma_r', None, 'Slum probability (2025)'),
                  (np.ma.masked_where(~valid, cls), CLS_CMAP, BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5, 5.5], 5),
                   'Flood susceptibility class'),
                  (np.ma.masked_invalid(hot), ListedColormap(['#e8e8e8', '#b2182b']), None,
                   'Exposure hotspot (p > 0.8 & very high)')]
        for j, (a, cm, nm, ttl) in enumerate(panels):
            ax = axs[i, j]
            im = ax.imshow(a, cmap=cm, norm=nm, vmin=None if nm else 0, vmax=None if nm else 1, interpolation='nearest')
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f'({chr(97 + i * 3 + j)}) {c.replace("_", " ")} — {ttl}', fontsize=8.5)
            if j == 1:
                cb = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02, ticks=[1, 2, 3, 4, 5])
                cb.ax.set_yticklabels(['Very low', 'Low', 'Moderate', 'High', 'Very high'], fontsize=7)
            elif j == 0:
                plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'Fig4_Exposure_Slum_Flood.png', dpi=300, bbox_inches='tight'); plt.close()
    print(f'✅ Figure 4 → {FIGURES_DIR / "Fig4_Exposure_Slum_Flood.png"}')

s3p = RESULTS_DIR / 'Supplementary' / 'Table_S3_Slum_Accuracy_SpatialBlocks.csv'
if s3p.exists():
    s3 = pd.read_csv(s3p)
    mets = ['Precision', 'Recall', 'F1', 'IoU', 'Kappa', 'MCC', 'Balanced_Acc', 'ROC_AUC']
    x = np.arange(len(s3)); wd = 0.8 / len(mets)
    fig, ax = plt.subplots(figsize=(10, 4))
    for k, m in enumerate(mets):
        ax.bar(x + (k - len(mets) / 2 + 0.5) * wd, s3[m], wd, label=m.replace('_', ' '))
    ax.set_xticks(x); ax.set_xticklabels(s3['City'].str.replace('_', ' ')); ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score (held-out spatial blocks)'); ax.legend(ncol=4, fontsize=8, frameon=False, loc='lower left')
    ax.set_title('(a) Slum classifier accuracy', fontsize=10, loc='left'); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.savefig(FIGURES_DIR / 'Fig6a_Slum_Accuracy.png', dpi=300, bbox_inches='tight'); plt.close()
    print(f'✅ Figure 6a → {FIGURES_DIR / "Fig6a_Slum_Accuracy.png"}')


In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 12 — Supplementary tables (S1: predictors retained after VIF; S2: LOCO; A1: settings)
# ═════════════════════════════════════════════════════════════════════
SUPP_DIR = RESULTS_DIR / 'Supplementary'
SUPP_DIR.mkdir(parents=True, exist_ok=True)

# Table S1 — predictors retained (✓) / removed by VIF screening (✗), per city
s1 = pd.DataFrame({'Predictor': TARGET_PREDICTORS})
for city, d in ALL_RESULTS.items():
    kept = set(d.get('features_kept', []))
    s1[city] = ['✓' if f in kept else '✗' for f in TARGET_PREDICTORS]
for f, why in EXCLUDED_PREDICTORS.items():
    s1.loc[len(s1)] = [f] + ['— (excluded: ' + why + ')'] * (s1.shape[1] - 1)
s1.to_csv(SUPP_DIR / 'Table_S1_Predictors_after_VIF.csv', index=False, encoding='utf-8-sig')

# Table S2 — leave-one-covariate-out (LOCO) importance
loco_p = TABLES_DIR / 'LOCO_all.csv'
if loco_p.exists():
    s2 = pd.read_csv(loco_p)
    s2.to_csv(SUPP_DIR / 'Table_S2_LOCO.csv', index=False, encoding='utf-8-sig')
else:
    print('⚠️ LOCO_all.csv not found — run Cells 4–5 first')

# Table A1 — tuning and training settings used in this run
a1 = pd.DataFrame([
    ('Bayesian search iterations', BAYES_N_ITER), ('Spatial CV block size (deg)', SPATIAL_BLOCK_DEG),
    ('Spatial hold-out fraction', TEST_FRAC), ('Samples per class (max)', N_PER_CLASS),
    ('DL epochs (max)', DL_EPOCHS), ('DL early-stopping patience', DL_PATIENCE),
    ('MC dropout passes', MC_DROPOUT_T), ('VIF threshold', 10), ('Random seed', 42),
], columns=['Setting', 'Value'])
a1.to_csv(SUPP_DIR / 'Table_A1_Settings.csv', index=False, encoding='utf-8-sig')
print('✅ Supplementary tables →', SUPP_DIR)
for f in sorted(SUPP_DIR.iterdir()):
    print('  ', f.name)


In [ ]:
# ═════════════════════════════════════════════════════════════════════
# CELL 10 — Run manifest (versions, settings, timings) + output inventory
# ═════════════════════════════════════════════════════════════════════
import sklearn, skopt
manifest = {
    'run_finished': datetime.now().isoformat(timespec='seconds'),
    'total_minutes': round((time.time() - T_START) / 60, 1),
    'python': platform.python_version(), 'device': str(device),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
                 'scikit-optimize': skopt.__version__, 'xgboost': xgb.__version__, 'lightgbm': lgb.__version__,
                 'catboost': __import__('catboost').__version__, 'torch': torch.__version__, 'shap': shap.__version__,
                 'rasterio': rasterio.__version__, 'geopandas': gpd.__version__},
    'settings': {'BAYES_N_ITER': BAYES_N_ITER, 'DL_EPOCHS': DL_EPOCHS, 'DL_PATIENCE': DL_PATIENCE,
                 'MC_DROPOUT_T': MC_DROPOUT_T, 'SPATIAL_BLOCK_DEG': SPATIAL_BLOCK_DEG, 'N_PER_CLASS': N_PER_CLASS,
                 'ALE': 'first-order, built-in (Apley & Zhu 2020), 10 quantile bins', 'THIN_GAP_M': THIN_GAP_M, 'VIF_threshold': 10, 'test_split': 'spatial-block hold-out, 20 % per class', 'seed': 42,
                 'class_method': CLASS_METHOD, 'excluded_predictors': EXCLUDED_PREDICTORS,
                 'fill_predictors': FILL_PREDICTORS, 'threshold_rule': 'Youden J on out-of-fold / inner-validation predictions',
                 'search_subdirs': SEARCH_SUBDIRS,
                 'min_tif_bytes': MIN_TIF_BYTES},
    'cities': {c: {k: str(v) for k, v in cfg.items()} for c, cfg in CITIES.items()},
    'per_city_minutes': {c: d.get('minutes') for c, d in ALL_RESULTS.items()},
}
(RESULTS_DIR / 'RUN_MANIFEST.json').write_text(json.dumps(manifest, indent=2, cls=NumpyEncoder))

print(f'📁 {RESULTS_DIR}')
for sub in sorted(p for p in RESULTS_DIR.iterdir() if p.is_dir()):
    files = sorted(sub.iterdir())
    print(f'  📁 {sub.name}/  ({len(files)} files)')
    for f in files[:60]:
        print(f'      {f.name}')
print(f'\n✅ PIPELINE COMPLETE — {manifest["total_minutes"]} min')